In [1]:
import torch

In [2]:
import numpy as np
import sklearn.metrics


class BaseExperiment:
    """Basic Experiment interface, with sane defaults"""

    def __init__(self):
        # use all tags
        self.selected_tags = MAEDataset.TAG_NAMES

        # drop hand-drawn instances
        self.instance_types = ["real", "simulated"]

        # use all classes
        self.tgt_events = MAEDataset.KNOWN_CLASSES
        self.num_classes = len(self.tgt_events)

    def raw_transform(self, event):
        """primary expert feature extraction from raw to processed data"""
        raise NotImplementedError()

    def balance(self, X, y, g, g_class):
        """by default, sample self.normal_balance * pseudo_normal_count true normals from X"""
        if not hasattr(self, "normal_balance"):
            return X, y, g
        if self.normal_balance == 0:
            return X, y, g

        # indices of samples from target events
        (tgt_idx,) = np.nonzero(g_class[g] != 0)

        # pseudo normal count
        pseudo_normal_count = np.count_nonzero(y[tgt_idx] == 0)

        # indices of samples from normal events
        (normal_idx,) = np.nonzero(g_class[g] == 0)
        normal_count = normal_idx.size

        balanced_count = min(self.normal_balance * pseudo_normal_count, normal_count)
        normal_idx = np.random.choice(normal_idx, balanced_count, replace=False)

        # compose final mask
        selected_idx = np.concatenate([normal_idx, tgt_idx])
        selected_idx.sort()

        # filter samples
        X = X[selected_idx]
        y = y[selected_idx]
        g = g[selected_idx]

        return X, y, g

    def fit(self, X, y):
        """fit preprocess pipeline to train data"""
        raise NotImplementedError()

    def transform(self, X, y):
        """apply preprocessing pipeline to data"""
        return X, y

    def fit_transform(self, X, y):
        self.fit(X, y)
        return self.transform(X, y)

    def metric_name(self):
        """returns metric name as a string. if sklearn has that scorer as a name can be used"""
        raise NotImplementedError()

    def metric_scorer(self):
        """returns a metric evaluation callback for RandomForest, usually sklearn `make_scorer'"""
        return sklearn.metrics.get_scorer(self.metric_name())

    def metric_lgbm(self):
        """by default, wrap an sklearn metric to the lgbm feval format"""
        name = self.metric_name()
        scorer = sklearn.metrics.get_scorer(name)
        _func = scorer._score_func
        _sign = scorer._sign == 1

        def _f(y_true, y_score):
            y_pred = y_score.reshape(-1, y_true.shape[0]).argmax(0)
            return name, _func(y_true, y_pred), _sign

        return _f

In [3]:
class TorchLabelStrategy:   # Estudar!!!!
    """
    Base class that just wraps applications of apply.
    Leverages pytorch unfold function (see above, apply windowing).

    (16/02/2024) Tensor.unfold(dimension, size, step) → Tensor. Returns a view of the original tensor which contains all slices of size 
    "size" from self tensor in the dimension "dimension".  Step between two slices is given by step.  An additional dimension of size "size" 
    is appended in the returned tensor.  
    
    (16/02/2024) Parameters of Tensor.unfold(dimension, size, step):
        dimension (int) – dimension in which unfolding happens
        size (int) – the size of each slice that is unfolded
        step (int) – the step between each slice

    (16/02/2024) Example:
    >>> x = torch.arange(1., 8)
    >>> x
    tensor([ 1.,  2.,  3.,  4.,  5.,  6.,  7.])
    >>> x.unfold(0, 2, 1)
    tensor([[ 1.,  2.],
            [ 2.,  3.],
            [ 3.,  4.],
            [ 4.,  5.],
            [ 5.,  6.],
            [ 6.,  7.]])
    >>> x.unfold(0, 2, 2)
    tensor([[ 1.,  2.],
            [ 3.,  4.],
            [ 5.,  6.]])
    
    (16/02/2024) pd.Series provide a one-dimensional labeled array with window of 100 or more, and apply according to "apply function" 
    (according used classes above, TorchBinaryMCLStrategy, TorchBinaryMRLStrategy, TorchOVAMCLStrategy, TorchOVATransientMCLStrategy, 
    TorchOVAMRLStrategy, TorchMulticlassMCLStrategy, and TorchMulticlassMRLStrategy). 
        
    (16/02/2024)For example, in TorchMulticlassMRLStrategy returns conditionally: If window is NAN, results in NAN, otherwise returns the  remainder of the number of elements from previous windows divided by 100.

    Variables 
        - labels -- Come from dataset.dataset.MAEDataset (line 184), and is the "Tag corresponding to instance label".
    """

    def __init__(self, window_size, stride=1, offset=0):  #stride - Number of samples between consecutive windows
        self.window_size = window_size # defined in modules of experiments (in stat, from 100 to 1000)
        self.stride = stride
        self.offset = offset

    def apply(self, y, event_type):
        raise NotImplementedError

    def __call__(self, labels, event_type):
        # store index
        index = labels.index

        # not enough samples for windowing, return empty
        if len(labels) < self.offset + self.window_size:  # If quantity of labels of some fail is less than size of window_size:
            out = pd.Series(name=MAEDataset.LABEL_NAME, dtype=np.float64)  # Create serie with these labels
            out.index.name = index.name # Select the index of these series
            return out

        # pass to pytorch as float (propagate nan)
        labels = torch.tensor(labels.values, dtype=torch.float32).squeeze() # create a multidimensional matrix of labels, but remove unitary elements

        # apply windowing (slices of window_size)
        y = labels[self.offset :].unfold(0, self.window_size, self.stride)  # y = labels.unfold(0, window_size, 1)
        index = index[self.offset :][self.window_size - 1 :: self.stride] # to extract (window size - 1) elements of index
          
        out = pd.Series(    # Constructing Series from a dictionary with an Index specified "index"
            name=MAEDataset.LABEL_NAME,
            data=self.apply(y, event_type), # Choose the label according to "apply fuction" 
            index=index,
            dtype=np.float64,
        )
        out.index.name = index.name
        return out

In [4]:
class TorchMulticlassMRLStrategy2(TorchLabelStrategy):
    """detect transients and faults,  most recent label
    Separate 0 class normal from 0 class pseudo_normal.
         """   
    
    _NAN = torch.tensor([np.nan]).float()
    
    def apply(self, y, event_type=None):
        
        last = y[:, -1]
        if last[1] == 0. and last[-1] >= 1.:
            last = torch.where(last == 0., 9.0, last.float()) 
        if last[1] == 0. and last[-1] == self._NAN:
            last = torch.where(last == 0., 9.0, last.float()) 
        
        return torch.where(last.isnan(), self._NAN, (last % 100).float()) 

In [5]:
"""Feature mapping strategies"""

import numpy as np
import pandas as pd
import scipy.stats as sp
import pywt

import torch


class StatisticalFeatureMapper:
    """
    Generates statistical descriptor for a window of our data

    """

    FEATURES = {
        "mean": lambda x: x.mean(),
        "std": lambda x: x.std(),
        "skew": lambda x: x.apply(sp.skew, raw=True),  # pandas skew is bad
        "kurt": lambda x: x.apply(sp.kurtosis, raw=True),  # kurtosis too
        "max": lambda x: x.max(),
        "min": lambda x: x.min(),
        "1qrt": lambda x: x.quantile(0.25),
        "med": lambda x: x.median(),
        "3qrt": lambda x: x.quantile(0.75),
    }

    def __init__(self, window_size, stride=1, offset=0):
        self.window_size = window_size
        self.stride = stride
        self.offset = offset

    def __call__(self, tags, event_type=None):
        # apply rolling window
        tag_names = tags.columns
        tags = tags.rolling(window=self.window_size)

        feats = []
        for f in StatisticalFeatureMapper.FEATURES:
            # apply feature mapping
            feat = StatisticalFeatureMapper.FEATURES[f](tags)
            # append feature_name to column
            feat = feat.rename(columns={t: f"{t}_{f}" for t in tag_names})
            feats.append(feat)

        feats = pd.concat(feats, axis="columns")
        return feats[self.offset :: self.stride]


class TorchStatisticalFeatureMapper:
    """
    PyTorch implementation of the statistical feature mapper
    """

    FEATURES = ["mean", "std", "skew", "kurt", "min", "1qrt", "med", "3qrt", "max"]

    def __init__(self, window_size, stride=10, offset=0, eps=1e-6):
        self.window_size = window_size
        self.stride = stride
        self.offset = offset
        self.eps = eps

    def __call__(self, tags, event_type=None):
        # preserve names and index
        columns = tags.columns
        index = tags.index

        # output names
        out_columns = [f"{t}_{f}" for f in self.FEATURES for t in columns]

        if len(tags) < self.offset + self.window_size:
            # not enough for a single window
            out = pd.DataFrame(
                columns=out_columns, dtype=np.float64
            )  # return empty dataframe
            out.index.name = index.name
            return out

        tags = torch.Tensor(tags.values).double()

        # apply windowing operation
        tags = tags[self.offset :].unfold(0, self.window_size, self.stride)
        index = index[self.offset :][self.window_size - 1 :: self.stride]

        # central moment
        std, mean = torch.std_mean(tags, dim=-1, unbiased=False)
        # centralized - standardized values
        cstags = (tags - mean.unsqueeze(-1)) / (std.unsqueeze(-1) + self.eps)
        # normalized moments
        skew = cstags.pow(3).mean(dim=-1)
        kurt = cstags.pow(4).mean(dim=-1)

        # quantiles
        quantiles = torch.tensor([0.00, 0.25, 0.50, 0.75, 1.00]).double()
        q0, q1, q2, q3, q4 = tags.quantile(quantiles, dim=-1)

        records = {}
        for i, t in enumerate(columns):
            records[f"{t}_mean"] = mean[:, i]
            records[f"{t}_std"] = std[:, i]
            records[f"{t}_skew"] = skew[:, i]
            records[f"{t}_kurt"] = kurt[:, i]
            records[f"{t}_min"] = q0[:, i]
            records[f"{t}_1qrt"] = q1[:, i]
            records[f"{t}_med"] = q2[:, i]
            records[f"{t}_3qrt"] = q3[:, i]
            records[f"{t}_max"] = q4[:, i]

        # fill dataframe in correct order
        out = pd.DataFrame.from_records(records, index=index, columns=out_columns)
        out.index.name = index.name  # also preserve index
        return out


class TorchEWStatisticalFeatureMapper:
    """PyTorch implementation of a truncated exponentially weighted
    statistical feature mapper"""

    FEATURES = [
        "ew_mean",
        "ew_std",
        "ew_skew",
        "ew_kurt",
        "ew_min",
        "ew_1qrt",
        "ew_med",
        "ew_3qrt",
        "ew_max",
    ]

    def __init__(self, window_size, decay, stride=10, offset=0, eps=1e-6):
        self.window_size = window_size
        self.stride = stride
        self.offset = offset
        self.eps = eps

        h = decay ** torch.arange(window_size, 0, step=-1)
        self.h = h / (h.abs().sum() + self.eps)  # L1 normalization

    def _E(self, X, dim=-1):
        """Take the exponentially weighted average trhough a dot in the specified dimension"""
        return torch.tensordot(X, self.h, dims=[[dim], [0]])

    def __call__(self, tags, event_type=None):
        # preserve names and index
        columns = tags.columns
        index = tags.index

        # output names
        out_columns = [f"{t}_{f}" for f in self.FEATURES for t in columns]

        if len(tags) < self.offset + self.window_size:
            # not enough for a single window
            out = pd.DataFrame(
                columns=out_columns, dtype=np.float64
            )  # return empty dataframe
            out.index.name = index.name
            return out

        tags = torch.tensor(tags.values).double()

        # apply windowing operation
        tags = tags[self.offset :].unfold(0, self.window_size, self.stride)
        index = index[self.offset :][self.window_size - 1 :: self.stride]

        # central moment
        mean = self._E(tags, dim=-1)
        std = self._E(torch.pow(tags - mean.unsqueeze(-1), 2)).sqrt()

        # centralized - standardized values
        cstags = (tags - mean.unsqueeze(-1)) / (std.unsqueeze(-1) + self.eps)
        # normalized moments
        skew = self._E(cstags.pow(3), dim=-1)
        kurt = self._E(cstags.pow(4), dim=-1)

        # quantiles
        quantiles = torch.tensor([0.00, 0.25, 0.50, 0.75, 1.00]).double()
        q0, q1, q2, q3, q4 = cstags.quantile(quantiles, dim=-1)

        records = {}
        for i, t in enumerate(columns):
            records[f"{t}_ew_mean"] = mean[:, i]
            records[f"{t}_ew_std"] = std[:, i]
            records[f"{t}_ew_skew"] = skew[:, i]
            records[f"{t}_ew_kurt"] = kurt[:, i]
            records[f"{t}_ew_min"] = q0[:, i]
            records[f"{t}_ew_1qrt"] = q1[:, i]
            records[f"{t}_ew_med"] = q2[:, i]
            records[f"{t}_ew_3qrt"] = q3[:, i]
            records[f"{t}_ew_max"] = q4[:, i]

        # fill dataframe in correct order
        out = pd.DataFrame.from_records(records, index=index, columns=out_columns)
        out.index.name = index.name  # also preserve index
        return out


class TorchWaveletFeatureMapper:
    """PyTorch implementation of the wavelet feature mapper"""

    def __init__(self, level, stride, offset=0):
        self.level = level
        self.window_size = 2**level
        self.stride = stride
        self.offset = offset

        impulse = np.zeros(self.window_size)
        impulse[-1] = 1
        hs = pywt.swt(impulse, "haar", level=self.level)
        H = np.stack([h[i] for h in hs for i in range(2)] + [impulse], axis=-1)

        self.feat_names = [
            f"{type_}{level}"
            for level in range(self.level, 0, -1)
            for type_ in ["A", "D"]
        ] + ["A0"]
        self.H = torch.tensor(H).double()

    def __call__(self, tags, event_type=None):
        # preserve names and index
        columns = tags.columns
        index = tags.index

        # output names
        out_columns = [f"{t}_{f}" for f in self.feat_names for t in columns]

        if len(tags) < self.offset + self.window_size:
            # not enough for a single window
            out = pd.DataFrame(
                columns=out_columns, dtype=np.float64
            )  # return empty dataframe
            out.index.name = index.name
            return out

        tags = torch.tensor(tags.values).double()

        # apply windowing operation
        tags = tags[self.offset :].unfold(0, self.window_size, self.stride)
        index = index[self.offset :][self.window_size - 1 :: self.stride]

        coeffs = torch.tensordot(tags, self.H, dims=([-1], [0]))

        records = {}
        for i, t in enumerate(columns):
            for j, f in enumerate(self.feat_names):
                records[f"{t}_{f}"] = coeffs[:, i, j]

        # fill dataframe in correct order
        out = pd.DataFrame.from_records(records, index=index, columns=out_columns)
        out.index.name = index.name  # also preserve index
        return out


class MixedMapper:
    """
    Join features of multiple mappers. Feature sizes must be consistent.

    """

    def __init__(self, *args):
        self.mappers = args

    def __call__(self, tags, event_type=None):
        return pd.concat([m(tags, event_type) for m in self.mappers], axis="columns")

In [6]:
""" definitions for experiment 3 module
    Multiclass classification
    Statistical+Wavelet features
    ***Most-Recent Label strategy, but splitting normal class of pseudonormal class (9)***
    Drop windows with NaN label
    Used in experiment 1 of the master's dissertation. 
"""

import numpy as np

# from sklearn.metrics import accuracy_score, get_scorer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA





MAX_LEVEL = 10


def sample(trial, *args, **kwargs):
    return Experiment(
        level=trial.suggest_int("level", 4, MAX_LEVEL, step=1),
        stride=trial.suggest_int("stride", 10, 10),
        n_components=trial.suggest_float("n_components", 0.9, 1.0),
        normal_balance=trial.suggest_int("normal_balance", 1, 10, step=1),
    )


class Experiment(BaseExperiment):
    """the docstring"""

    def __init__(
        self,
        level,
        stride,
        n_components,
        normal_balance,
        *args,
        **kwargs,
    ):
        super().__init__()

        # save params
        self.window_size = 2**level
        self.level = level
        self.n_components = n_components
        self.stride = stride
        self.normal_balance = normal_balance

        self._init_raw_mappers()
        self._init_preprocessor()

    def raw_transform(self, event, transient_only=False, no_nans=True):
        # filter tags and set zeros to nans
        tags = event["tags"][self.selected_tags].replace(0, np.nan)
        labels = event["labels"]
        event_type = event["event_type"]
        file_name = event["file_name"]

        # to trim stablished fault if has transient
        if transient_only and MAEDataset.TRANSIENT_CLASS[event_type]:   # Seleciona somente os eventos que possuem fase transiente (1,2,5,6,7,8)
            transients = labels.values != event_type
            tags = tags[transients]
            labels = labels[transients]

        features = self._feature_mapper(tags, event_type)
        labels = self._label_mapper(labels, event_type)

        # drop windows with NaN label
        if no_nans:
            notnan = labels.notna()
            features = features[notnan]
            labels = labels[notnan]

        return features, labels, event_type,file_name

    def metric_name(self):
        return "balanced_accuracy"  # alterado em 11/12/23 para balanced_accuracy

    def metric_rf(self):
        return get_scorer("balanced_accuracy")

    def metric_lgbm(self):
        def acc(preds, train_data):
            preds_ = np.argmax(np.reshape(preds, (self.num_classes, -1)), axis=0)
            return "balanced_accuracy", accuracy_score(train_data.get_label(), preds_), True

        return acc

    
    def fit(self, X, y):
        y = self._label_encoder.fit_transform(y)   # Encode target labels with value between 0 and n_classes-1.
        # Standardize features by removing the mean and scaling to unit variance.  Fit to data, then transform it.
        X = self._scaler.fit_transform(X) 
        # Replace missing values using a descriptive statistic (mean) along each column.  Fit to data, then transform it.
        X = self._imputer.fit_transform(X)
        # Fit the model with X - Just Singular Value Decomposition to project it to a lower dimensional space.                         
        self._pca.fit(X, y)  

    def transform(self, X, y=None):  # Used in tune_lgbm, score_model function (dimensionality_reduction step)
        y = self._label_encoder.transform(y)
        # Standardize features by removing the mean and scaling to unit variance.  Perform standardization by centering and scaling.
        X = self._scaler.transform(X)
        # Replace missing values using a descriptive statistic (mean) along each column.  Impute all missing values in X.
        X = self._imputer.transform(X)
        # Apply dimensionality reduction to X.
        X = self._pca.transform(X)  
        return X, y

    def _init_raw_mappers(self):
        offset = 2**MAX_LEVEL - self.window_size
        wavelet_mapper = TorchWaveletFeatureMapper(
            level=self.level, stride=self.stride, offset=offset
        )
        stats_mapper = TorchStatisticalFeatureMapper(
            window_size=2**self.level, stride=self.stride, offset=offset
        )

        self._feature_mapper = MixedMapper(stats_mapper, wavelet_mapper)

        self._label_mapper = TorchMulticlassMRLStrategy2(
            window_size=self.window_size,
            stride=self.stride,
            offset=offset,
        )

    def _init_preprocessor(self):
        # z-score
        self._scaler = StandardScaler()
        # remove nans
        self._imputer = SimpleImputer(strategy="mean")
        # pca
        self._pca = PCA(n_components=self.n_components, whiten=True)  # n_components from 0.9 to 1.0
        """
        sklearn.decomposition.PCA:
        Linear dimensionality reduction using Singular Value Decomposition of the data to project it to a lower dimensional space. The 
        input data is centered but not scaled for each feature before applying the SVD.
        
        The exact full SVD is computed and optionally truncated afterwards.
       
        If 0 < n_components < 1 and svd_solver == 'full', select the number of components such that the amount of variance that needs 
        to be explained is greater than the percentage specified by n_components.
        Whitening will remove some information from the transformed signal (the relative variance scales of the components) but can 
        sometime improve the predictive accuracy of the downstream estimators by making their data respect some hard-wired assumptions.
        """ 
        # label encoder for multiclass
        self._label_encoder = LabelEncoder()

In [7]:
# coding: utf-8
"""Basic dataset definitions"""

from collections import namedtuple
from pathlib import Path

import numpy as np
import pandas

from joblib import Parallel, delayed


class MAEDataset:
    """Load all files and return transformed glob

    Most of this class deals with selecting which event '.csv's should be loaded,
        and feeding each 'raw' csv to some feature mapper.

    * Constructor arguments:
        - **root_dir: STRING** -- base location of events separated by event type

        - **tgt_class: LIST** -- codes of which event types should be used

        - **selected_tags: STRING LIST** -- names of columns that should be used on
            feature extraction, i.e. ["T-TPT", "QGL"]

        - **instance_types: STRING LIST** -- types of instances that should be used list containing
            one or more of {"real", "simulated", "drawn"}

        - **feature_mapper: CALLABLE**
        (raw_tags: DataFrame,  raw_labels: DataFrame) ->
        (feature: DataFrame x label: DataFrame) --
        deals with transforming the raw data from a single event,
        to be used during training/evaluation. Outputs from each event concatenated
        afterwards.

        - **n_jobs: INT** -- number of processes to use

        - **events**

    * Important fields:

        - **self.X: [N_SAMPLE x N_FEATURES]** ndarray created by concatenating every feature output from feature mapper

        - **self.Y: [N_SAMPLE]** ndarray created by concatenating every label output from feature mapper

        - **self.feature_names: [N_FEATURES]** ndarray as output by the feature mapper

        - **self.g_len: [N_EVENTS]** number of samples output by the feature mapper for each event

        - **self.g_class: [N_EVENTS]** list denoting the origin of the event


    * Methods:

        - **_is_instance_type(type_)**

        - **_make_set()**

        - **process(fname, event_type)**

    """

    # Tag corresponding to instance label
    LABEL_NAME = "class"

    # Tag corresponding to index
    INDEX_NAME = "timestamp"

    # Fault description
    CLASS_NAMES = {
        0: "NORMAL",
        1: "ABRUPT_INCREASE_OF_BSW",
        2: "SPURIOUS_CLOSURE_OF_DHSV",
        3: "SEVERE_SLUGGING",
        4: "FLOW_INSTABILITY",
        5: "RAPID_PRODUCTIVITY_LOSS",
        6: "QUICK_RESTRICTION_IN_PCK",
        7: "SCALING_IN_PCK",
        8: "HYDRATE_IN_PRODUCTION_LINE",
    }

    # List of used classes
    KNOWN_CLASSES = list(CLASS_NAMES.keys())

    # Transient properties of events
    TRANSIENT_CLASS = {
        0: False,
        1: True,
        2: True,
        3: False,
        4: False,
        5: True,
        6: True,
        7: True,
        8: True,
    }

    # List of known data tags
    TAG_NAMES = [
        "P-PDG",
        "P-TPT",
        "T-TPT",
        "P-MON-CKP",
        "T-JUS-CKP",
        "P-JUS-CKGL",
        "T-JUS-CKGL",
        "QGL",
    ]

    def __init__(
        self,
        transformed_events=None,
        events=None,  # either pass in preloaded events or the root directory
        root_dir=None,
        tgt_events=[],  # which event_types to load
        instance_types=[],  # simulated and or real and or drawn
        feature_mapper=tuple,  # transformer from event to features
        n_jobs=-1,
    ):
        """
        Load and process dataset using supplied strategies.
        """

        # save parameters
        self.root_dir = root_dir
        self.tgt_events = tgt_events
        self.instance_types = instance_types
        self.n_jobs = n_jobs
        self.feature_mapper = feature_mapper

        # Call the heavy load _make_set passing the (maybe Null) events
        self._make_set(events)

    def _instance_type(fname):
        """
        Detects if instance type is selected.

        * Parameters:
            - **fname**: STRING - name of the instance file

        * Returns:
            - **STRING** - string representing the instance type of the input file name

        """
        if fname.startswith("OLGA"):
            return "simulated"
        elif fname.startswith("DESENHADA"):
            return "drawn"
        else:
            return "real"

    def load_events(data_root, n_jobs=-1):
        """
        Scan data_root for raw files and return dict. useful for preloads.

        * Parameters:
            - **data_root: STRING** - base location of events separated by event type

        * Returns:
            - **events**: [LIST] - Optional list of preloaded events
        """

        def _read(tgt, fname):
            """
            Return a dict with the summary of a target.

            * Parameters:
                - **tgt: STRING** - Target location

            * Returns:
                - **fname**: STRING - Filename
            """
            df = pandas.read_csv(
                fname,
                index_col=MAEDataset.INDEX_NAME,
                header=0,
                parse_dates=True,
                memory_map=True,
            )
            tags = df[MAEDataset.TAG_NAMES]
            labels = df[MAEDataset.LABEL_NAME]
            return df 
            # return {
            #     "file_name": str(fname.relative_to(tgt)),
            #     "tags": tags,
            #     "labels": labels,
            #     "event_type": int(str(tgt.relative_to(data_root))),
            # }

        data_root = Path(data_root)
        target_dirs = [
            d for d in data_root.iterdir() if d.match("[0-8]")
        ]  # filter directories with classes
        tasks = [(tgt, fname) for tgt in target_dirs for fname in tgt.glob("*.csv")]
        with Parallel(n_jobs) as p:
            events = p(delayed(_read)(*t) for t in tasks)
        return events

    def transform_events(
        events, raw_mapper, tgt_events=None, instance_types=None, n_jobs=-1
    ):
        """
        Apply raw_mapper to list of events, filtering by target events and instance types
        """
        if tgt_events is not None:
            events = [e for e in events if (e["event_type"] in tgt_events)]
        if instance_types is not None:
            events = [
                e
                for e in events
                if (MAEDataset._instance_type(e["file_name"]) in instance_types)
            ]

        with Parallel(n_jobs) as p:
            return p(delayed(raw_mapper)(e) for e in events)

    def gather(transformed_events):
        Dataset = namedtuple("Dataset", ["X", "y", "g", "g_class","file"])

        X = pandas.concat([e[0] for e in transformed_events], axis="index").values
        y = pandas.concat([e[1] for e in transformed_events], axis="index").values
        sizes = [e[1].size for e in transformed_events]

        g = np.repeat(np.arange(len(sizes)), sizes)
        g_class = np.array([e[2] for e in transformed_events])
        file = np.array([e[3] for e in transformed_events])

        return Dataset(X=X, y=y, g=g, g_class=g_class,file=file)

    def _make_set(self, events=None):
        """
        Loads all instances of target classes from the desired types, transforming the raw data to obtain its
        features by calling the *feature_mapper()* method for each instance.

        * Parameters:
            - **events**: [LIST] - Optional list of preloaded events
        """

        # load if not preloaded
        if events is None:
            events = MAEDataset.load_events(self.root_dir)

        # filter events
        def _should_keep(e):
            return (
                MAEDataset._instance_type(e["file_name"]) in self.instance_types
            ) and (e["event_type"] in self.tgt_events)

        events = [e for e in events if _should_keep(e)]

        feature_names = []
        X = []
        y = []
        g_len = []
        g_class = []

        with Parallel(self.n_jobs) as p:
            for x_, y_, et_ in p(
                delayed(self.feature_mapper)(event) for event in events
            ):
                # save feature names to check for consistency
                feature_names.append(x_.columns)

                # save features and labels
                X.append(np.double(x_.values))
                y.append(np.uint8(y_.values.ravel()))

                # save length of each event output
                g_len.append(y_.size)

                # save origin of each event
                g_class.append(et_)

        # feature name consistency check
        assert all(np.all(feature_names[0] == c) for c in feature_names)

        # glob features and labels
        self.X = np.concatenate(X, axis=0)
        self.y = np.concatenate(y, axis=0).ravel()

        # convert to np
        self.g_len = np.array(g_len)
        self.g_class = np.array(g_class)
        self.g = np.repeat(np.arange(self.g_len.size), self.g_len)

        self.feature_names = np.array(feature_names[0].to_list())

In [8]:
"""Copyright [2024] [Antonio Alberto Moreira de Azevedo]

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License."""

#! python3confusion_matrix
# -*- coding: utf-8 -*-
import os
import importlib
import warnings
import pickle
import tempfile
import json
import datetime
import logging

import numpy as np
import lightgbm as lgb
import optuna
import mlflow

import os
# import dagshub
from getpass import getpass
from mlflow import MlflowClient

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, recall_score
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.decomposition import PCA
import click

import joblib

# import dask.distributed

import seaborn as sns
import matplotlib.pyplot as plt



def warn(*args, **kwargs):
    pass


warnings.warn = warn

np.seterr(divide="ignore", invalid="ignore")


logger = logging.getLogger("tune_lgbm")
logger.setLevel(logging.DEBUG)

ch = logging.StreamHandler()
ch.setLevel(logging.INFO)
logger.addHandler(ch)

formatter = logging.Formatter("[%(asctime)s - %(name)s - %(levelname)s] %(message)s")
ch.setFormatter(formatter)

lgb.register_logger(logger)

####################################################
# some helper functions


def metric_suite(y_true, y_score, model):   
    """compute many metrics using y_pred (predict_proba)"""
    y_pred = np.argmax(y_score, -1) #Returns the indices of the maximum values along an last axis.  y_score can be result of training or test.
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_micro": precision_score(y_true, y_pred, average="micro"),
        "precision_macro": precision_score(y_true, y_pred, average="macro"),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted"),
        "recall_micro": recall_score(y_true, y_pred, average="micro"),
        "recall_macro": recall_score(y_true, y_pred, average="macro"),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted"),
        "f1_micro": f1_score(y_true, y_pred, average="micro"),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
        "roc_auc_ovr_macro": roc_auc_score(
            y_true, y_score, average="macro", multi_class="ovr"
        ),
        "roc_auc_ovr_weighted": roc_auc_score(
            y_true, y_score, average="weighted", multi_class="ovr"
        ),
        "roc_auc_ovo_macro": roc_auc_score(
            y_true, y_score, average="macro", multi_class="ovo"
        ),
        "roc_auc_ovo_weighted": roc_auc_score(
            y_true, y_score, average="weighted", multi_class="ovo"
        ),
    }
   
    cm = {                 
        "raw": confusion_matrix(y_true, y_pred, normalize=None),  # Compute confusion matrix to evaluate the accuracy of a classification. 
        "normalized": confusion_matrix(y_true, y_pred, normalize="true"), 
    }

    return {"metrics": metrics, "confusion_matrix": cm}


def gather_results(results):
    """combine list of dicts of several runs into dict of lists"""
    return {
        "scores": [r["score"] for r in results],
        "metrics": {
            m: [r["metrics"][m] for r in results] for m in results[0]["metrics"].keys()
        },
        "confusion_matrices": {
            m: [r["confusion_matrix"][m] for r in results]
            for m in results[0]["confusion_matrix"].keys()
        },
    }


def log_result(result):   # result = score_model(train_set, test_set, best_experiment, best_model, config["n_jobs"], 489 line ("testing" of mlflow); score_model function is stated in 188 line
    """log result dictionary for a single run to mlflow"""

    # log target metric
    mlflow.log_metric("score", result["score"])

    # log aux metrics
    mlflow.log_metrics(result["metrics"])
    
    fig, ax = plot_confusion_matrix(result["confusion_matrix"]["normalized"])
    ax.set_title("Confusion matrix")
    mlflow.log_figure(fig, "confusion_matrix.png")
    plt.close(fig)

    # store model and experiment
    with tempfile.TemporaryDirectory() as tmp_dir:
        with open(os.path.join(tmp_dir, "model.pkl"), "wb") as f:
            pickle.dump(result["model"], f)
        mlflow.log_artifact(f.name)

        with open(os.path.join(tmp_dir, "experiment.pkl"), "wb") as f:
            pickle.dump(result["experiment"], f)
        mlflow.log_artifact(f.name)

        with open(os.path.join(tmp_dir, "result.pkl"), "wb") as f:   
            pickle.dump(result["confusion_matrix"]["normalized"], f)
        mlflow.log_artifact(f.name)
        
def log_results(results):    # results = cross_val_score(events, experiment, model, config["num_splits"], config["n_jobs"]),  280 line, variable of "hyperparameter_search" function
    """log gathered results dictionary for a cross-validated run to mlflow"""
    # log target metric
    mlflow.log_metric("score-avg", np.mean(results["scores"]))
    mlflow.log_metric("score-std", np.std(results["scores"]))

    # log aux metrics
    for k, v in results["metrics"].items():
        mlflow.log_metric(f"{k}-avg", np.mean(v))
        mlflow.log_metric(f"{k}-std", np.std(v))

    # plot and log confusion matrices 
    fig, ax = plot_confusion_matrix(
        np.sum(results["confusion_matrices"]["raw"], axis=0), normalize=True
    )
    ax.set_title("Overall confusion matrix")
    mlflow.log_figure(fig, "overall_confusion_matrix.png")
    plt.close(fig)

    fig, ax = plot_confusion_matrix(
        np.mean(results["confusion_matrices"]["normalized"], axis=0),
        np.std(results["confusion_matrices"]["normalized"], axis=0),
    )
    ax.set_title("Average confusion matrix")
    mlflow.log_figure(fig, "averaged_confusion_matrix.png")
    plt.close(fig)


def plot_confusion_matrix(cm, std=None, normalize=False):
    """generate confusion matrix visualization"""
    if normalize:
        cm = cm / cm.sum(1, keepdims=True)  # normalize over rows

    if std is not None:
        annot = [
            [rf"${v:.3f}\pm{s:.3f}$" for v, s in zip(vr, sr)] for vr, sr in zip(cm, std)  # Sets decimal places of the matrix
        ]
    else:
        annot = [[f"${v:.3f}$" for v in vr] for vr in cm]     #f"${v:.2f}$"
    
    fig, ax = plt.subplots(figsize=(9, 9))  # alterado para 8,8
    sns.heatmap(cm, cmap="viridis", annot=annot, fmt="", square=True, ax=ax)  # Sets decimal places of encoded matrix
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    return fig, ax


###########################
# main functions


def score_model(train_set, test_set, experiment, model, n_jobs=-1):
    """train and evaluate an experiment/model pair"""
    # split 20% for early stopping set
    Xt, Xs, yt, ys, gt, gs = train_test_split(
        train_set.X,
        train_set.y,
        train_set.g,
        test_size=0.2, # esta função separa base de treino para fazer o "early stop"
        random_state=0,
        # stratify=train_set.y,
    )
    print(f"train_set after splitted (80 %): X(shape) ={Xt.shape} /  y(shape) = {yt.shape} /  glen(shape) = {gt.shape}")  # Para ver a dimensão pós separação
    print(f"assess_set after splitted for early-sttoping (20%): X(shape) ={Xs.shape} /  y(shape) = {ys.shape} /  glen(shape) = {gs.shape}")  # Para ver a dimensão pós separação
    # rebalance train set
    Xt, yt, _ = experiment.balance(Xt, yt, gt, train_set.g_class)   # ****It must be commented when run with pseudo_normal splitted ****  !!!
    print(f"train_set after balanced: X(shape) ={Xt.shape} / y(shape) = {yt.shape} / glen(shape) = {gt.shape}")  # para ver dimensão pós bal de  eventos 0 normal e 0 de falha

    # preprocess and fit - 
    """
    independent centering and scaling of the features, all missing values are replaced with 0 (the new mean value), 
    and a  dimensionality-reducing step is performed
    """
    print("-------------------------------------experiment.fit-------------------------------------------")
    Xt, yt = experiment.fit_transform(Xt, yt) #Fit the model with X (Singular Value Decomposition to project it to a lower dimens space
    print(f"train_set after dimensionality_reduction step: X(shape) ={Xt.shape} / y(shape) = {yt.shape}")
    print('-------------Concluded preprocess and data transf for training------------------------')  
    Xs, ys = experiment.transform(Xs, ys) # Apply dimensionality reduction to X 
    print(f"assess_set after dimensionality_reduction step for early-stopping: Xs(shape) ={Xs.shape} / ys(shape) = {ys.shape}")
    
    
    logger.info(f"model.fit -- X.shape={Xt.shape}")
    fit_cb = [
        lgb.early_stopping(
            50,
            verbose=False,
        ),
        lgb.log_evaluation(10),
    ]
    model.fit(Xt, yt, eval_set=[(Xs, ys)], callbacks=fit_cb) # lightgbm.LGBMClassifier.fit - Build a gradient boosting model from the training set (X, y).
    print('-------------  Concluded training ----------------------------')  
    
    XT, yT = experiment.transform(test_set.X, test_set.y)
    print(f"validation_set or test_set after dimensionality_reduction: X(shape) ={XT.shape} / y(shape) = {yT.shape}")
    # compute target metric
    score = experiment.metric_scorer()(model, XT, yT)
   
    # collect auxiliary metrics
    print("---------------------------------------------------------------------------------------------------")
    """ 
    https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.LGBMClassifier.html
    lightgbm.LGBMClassifier.predict_proba(X[, raw_score, ...]) --> Return the predicted probability for each class for each sample.
    """
    y_score = model.predict_proba(XT)  #Return the predicted probability for each class for each sample.
    print(f"size of predicted probability for each class for each sample ={y_score.shape}")
    return {
        "model": model,
        "experiment": experiment,
        "score": score,
        **metric_suite(yT, y_score, model),
    }
    y_score2 = model.score(XT, yT)  
    print(f"mean accuracy on the given test data and labels ={y_score2[0]}")

def cross_val_score(events, experiment, model, num_splits, n_jobs=-1):
    """train and evaluate an experiment/model pair across several splits"""
    # preprocess all events
    transformed_events = MAEDataset.transform_events(
        events,
        experiment.raw_transform, # call _feature_mapper variable, found in _init_raw_mappers function of the selected experiment!!!!
        instance_types=experiment.instance_types,
        tgt_events=experiment.tgt_events,
        n_jobs=n_jobs,
    )
    print('---------------------------------------Concluding feature extraction ---------------------------------------------')
    
    # gather event types for stratification 
    event_types = [e[2] for e in transformed_events]
    logger.info(f"number of instances(events) in both training and validation = {np.array(event_types).shape}") # Displaying selected events by indexing subdirectories 
    # (excluindo o timestamp e caracteríticas)
    
    results = []
    for train, test in StratifiedKFold(num_splits).split(      
        transformed_events, event_types
    ):
        train_set = MAEDataset.gather([transformed_events[i] for i in train])
        X_train_raw = train_set.X
        print(f"train_set (80%) --> X = {np.array(X_train_raw).shape}", "g_class =", np.array(train_set.g_class).shape) # Exibindo o conjunto de treinamento
        test_set = MAEDataset.gather([transformed_events[i] for i in test])
        X_valid_raw = test_set.X 
        print(f"validation_set (20%) --> X = {np.array(X_valid_raw).shape}", "g_class =", np.array(test_set.g_class).shape) # Exibindo o conjunto de validação
        results.append(score_model(train_set, test_set, experiment, model, n_jobs))
        

    # gather result
    return gather_results(results)


def hyperparameter_search(events, experiment_sampler, model_sampler, config):
    # single trial is a k-fold
    def objective(trial):   # A custom objective function for the objective parameter of LightGBM classifier
        experiment = experiment_sampler(trial)
        model = model_sampler(trial)   # model_sampler = lightgbm_sampler, 467 line in tune function

        with mlflow.start_run(nested=True, run_name=f"trial - {trial.number} - cv"):
            mlflow.log_params(trial.params)
            results = cross_val_score(
                events, experiment, model, config["num_splits"], config["n_jobs"]
            )
            log_results(results)

        # optimize mean of target metric
        return np.mean(results["scores"])

    # create study with optional fixed hyperparameters
    study = optuna.create_study(
        sampler=optuna.samplers.PartialFixedSampler(
            config["fixed_params"],
            optuna.samplers.TPESampler(
                multivariate=True, warn_independent_sampling=False
            ),
        ),
        direction="maximize",
    )

    # load additional options
    for k, v in config["user_attrs"].items():
        study.set_user_attr(k, v) # (Optuna function) Set user attributes to the trial. 

    with mlflow.start_run(nested=True, run_name="tuning"):

        def mlflow_callback(study, trial):
            mlflow.log_metric("score-avg", trial.value, trial.number)

        study.optimize(objective, config["num_trials"], callbacks=[mlflow_callback])
        mlflow.log_params(study.best_params)
        mlflow.log_metric("best", study.best_value)

        # log study as artifact for later analysis
        with tempfile.TemporaryDirectory() as tmp_dir:
            with open(os.path.join(tmp_dir, "study.pkl"), "wb") as f:
                pickle.dump(study, f)
            mlflow.log_artifact(f.name)
    return study


def lightgbm_sampler(trial):
    return lgb.LGBMClassifier(
        boosting_type="gbdt",  # traditional Gradient Boosting Decision Tree
        n_estimators=500,  # Number of boosted trees to fit.
        learning_rate=0.1, 
        is_unbalance=True,
        subsample_freq=1,
        verbose=-1,
        metrics=["multi_error"],
        subsample=trial.suggest_float("subsample", 0.1, 1.0, step=0.05),
        colsample_bytree=trial.suggest_float("feature_fraction", 0.1, 1.0, step=0.05),
        reg_alpha=trial.suggest_float("lambda_l1", 1e-5, 10, log=True),
        reg_lambda=trial.suggest_float("lambda_l2", 1e-5, 10, log=True),
        num_leaves=trial.suggest_int("num_leaves", 4, 128, step=1),
    )


def parse_json(ctx, self, value):
    return json.loads(value)


@click.group()
@click.pass_context
def cli(ctx, **kwargs):
    ctx.ensure_object(dict)
    ctx.obj.update(kwargs)

@cli.command()
@click.option("-t", "--train-root", type=click.Path(exists=True))
@click.option("-T", "--test-root", type=click.Path(exists=True))
@click.option("-e", "--experiment-name", type=click.STRING)
@click.option("-n", "--num-trials", type=click.INT, default=100)
@click.option("-s", "--num-splits", type=click.INT, default=5)

@click.option("-j", "--n-jobs", type=click.INT, default=-1)
@click.option("--fixed-params", type=click.STRING, default="{}", callback=parse_json)
@click.option("--user-attrs", type=click.STRING, default="{}", callback=parse_json)
@click.pass_context
def tune(ctx, **kwargs):
    # grab all cli params
    config = {**ctx.obj, **kwargs}

    # _ = dask.distributed.Client(n_workers=config["n_jobs"], processes=True)
    with joblib.parallel_backend("loky", n_jobs=config["n_jobs"]):
        # preload events
        train_events = MAEDataset.load_events(config["train_root"], config["n_jobs"])

        # select samplers
        model_sampler = lightgbm_sampler  # lightgbm_sampler function, 386 line
        experiment_sampler = importlib.import_module(config["experiment_name"]).sample
        
        os.environ['MLFLOW_TRACKING_USERNAME'] = input('[user name]')
        os.environ['MLFLOW_TRACKING_PASSWORD'] = getpass('[getpass]')
        os.environ['MLFLOW_TRACKING_PROJECTNAME'] = input('[project name]')

        # Use the DagsHub client to setup connection information for MLflow
        # dagshub.init(repo_owner='betomazevedo39', repo_name='3W', mlflow=True)

        # mlflow.set_tracking_uri('https://dagshub.com/[user name]/3W.mlflow') # reference to the Tracking server’s address
        
        # create experiment
        mlflow.set_experiment(datetime.datetime.now().strftime("%Y%m%d_%H%M_tuning"))

        with mlflow.start_run(run_name="testing"):
            mlflow.log_params(config)

            # find best hyper-params
            study = hyperparameter_search(
                train_events, experiment_sampler, model_sampler, config
            )

            # train model with best params
            best_trial = study.best_trial
            best_experiment = experiment_sampler(best_trial)
            best_model = model_sampler(best_trial)

            mlflow.log_params(best_trial.params)

            # map and gather tranining set
            transformed_train_events = MAEDataset.transform_events(
                train_events,
                best_experiment.raw_transform,
                instance_types=best_experiment.instance_types,
                tgt_events=best_experiment.tgt_events,
                n_jobs=config["n_jobs"],
            )
            train_set = MAEDataset.gather(transformed_train_events)
            print(' /////////////////////////////////////////////////////////////\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')     
            print(f"train_set (70%) ={np.array(train_set.X).shape}")     # Displaying train set without k-fold
            
            # map and gather test set
            test_events = MAEDataset.load_events(config["test_root"], config["n_jobs"])
            transformed_test_events = MAEDataset.transform_events(
                test_events,
                best_experiment.raw_transform,
                instance_types=best_experiment.instance_types,
                tgt_events=best_experiment.tgt_events,
                n_jobs=config["n_jobs"],
            )
            test_set = MAEDataset.gather(transformed_test_events)
            print(f"test_set (30%) ={np.array(test_set.X).shape}")     # Displaying test set
            result = score_model(
                train_set, test_set, best_experiment, best_model, config["n_jobs"]
            )
            # gather event types for stratification 
            event_types2 = [e[2] for e in transformed_test_events]
            logger.info(f"number of instances(events) in both training and validation = {np.array(event_types2).shape}") # Displaying selected events by indexing subdirectories
           
            log_result(result)
            


In [9]:
def load_instance(instance_path):
    try:
        # well, instance_id = instance_path.stem.split('_')
        df = pd.read_csv(instance_path)
        # assert (df.columns == columns).all(), 'invalid columns in the file {}: {}'\
        #     .format(str(instance_path), str(df.columns.tolist()))
        return df
    except Exception as e:
        raise Exception('error reading file {}: {}'.format(instance_path, e))

In [10]:
def load_and_downsample_instances(instances, downsample_rate, source, instance_id):
    df_instances = pd.DataFrame()
    for _, row in instances.iterrows():
        diretorio, instance_path = row
        df = load_instance(instance_path).iloc[::downsample_rate, :]
        print(instance_path)
        df = df[[  "timestamp", "P-PDG",
        "P-TPT",
        "T-TPT",
        "P-MON-CKP",
        "T-JUS-CKP",
        "P-JUS-CKGL",
        "T-JUS-CKGL",
        "QGL",
        "class"]]
        df['instance_id'] = instance_id
        df['Diretorio'] = diretorio
        # df['Source'] = source
        instance_id += 1
        df_instances = pd.concat([df_instances, df])
    return df_instances.reset_index(drop=True), instance_id

In [11]:
def get_instances(data_path, 
                                         real, simulated, drawn):
    instances = pd.DataFrame(class_and_file_generator(data_path,
                                                      real=real,
                                                      simulated=simulated, 
                                                      drawn=drawn),
                             columns=['class_code', 'instance_path'])
    return instances

In [12]:
def class_and_file_generator(data_path, real=False, simulated=False, drawn=False):
    for class_path in data_path.iterdir():
        if class_path.is_dir():
            if class_path.stem != 'folds':
                if class_path.stem.isdigit():  # Verifica se é número
                    class_code = int(class_path.stem)
                    for instance_path in class_path.iterdir():
                        if instance_path.suffix == '.csv':
                            if (simulated and instance_path.stem.startswith('SIMULATED')) or \
                               (drawn and instance_path.stem.startswith('DRAWN')) or \
                               (real and (not instance_path.stem.startswith('SIMULATED')) and \
                                        (not instance_path.stem.startswith('DRAWN'))):
                                yield class_code, instance_path
                else:
                    # Ignorar pastas que não são numéricas, como .ipynb_checkpoints
                    pass


In [13]:
data_path = Path(r'./treino')

instances_treino = get_instances(data_path, 
                                                      real=True,
                                                      simulated=True, 
                                                      drawn=False)

data_path = Path(r'./test')

instances_test = get_instances(data_path, 
                                                      real=True,
                                                      simulated=True, 
                                                      drawn=False)

In [14]:
downsample_rate = 60
instance_id = 1
df_treino, instance_id  = load_and_downsample_instances(instances_treino,
                                                            downsample_rate,
                                                            'real', 
                                                            instance_id)

df_test, instance_id  = load_and_downsample_instances(instances_test,
                                                            downsample_rate,
                                                            'real', 
                                                            instance_id)

treino/2/WELL-00012_20170320033022.csv
treino/2/WELL-00011_20140726180015.csv
treino/2/WELL-00011_20140921200031.csv
treino/2/WELL-00011_20140929170028.csv
treino/2/SIMULATED_00004.csv
treino/2/SIMULATED_00005.csv
treino/2/SIMULATED_00003.csv
treino/2/WELL-00011_20140530100015.csv
treino/2/SIMULATED_00002.csv
treino/2/WELL-00011_20140929220121.csv
treino/2/WELL-00011_20141005170056.csv
treino/2/WELL-00011_20141006160121.csv
treino/2/WELL-00011_20140928100056.csv
treino/2/WELL-00003_20170728150240.csv
treino/2/SIMULATED_00014.csv
treino/2/SIMULATED_00011.csv
treino/2/WELL-00003_20141122214325.csv
treino/2/WELL-00009_20170313160804.csv
treino/2/SIMULATED_00015.csv
treino/2/SIMULATED_00016.csv
treino/2/WELL-00002_20131104014101.csv
treino/2/SIMULATED_00007.csv
treino/2/WELL-00011_20140515110134.csv
treino/2/SIMULATED_00001.csv
treino/2/SIMULATED_00008.csv
treino/2/WELL-00003_20180206182917.csv
treino/3/SIMULATED_00005.csv
treino/3/SIMULATED_00009.csv
treino/3/SIMULATED_00014.csv
treino/3/

In [15]:
df_treino['class'] = df_treino['class'].fillna(0)
df_treino.loc[df_treino['class'] == 101, 'class'] = 1
df_treino.loc[df_treino['class'] == 102, 'class'] = 2
df_treino.loc[df_treino['class'] == 103, 'class'] = 3
df_treino.loc[df_treino['class'] == 104, 'class'] = 4
df_treino.loc[df_treino['class'] == 105, 'class'] = 5
df_treino.loc[df_treino['class'] == 106, 'class'] = 6
df_treino.loc[df_treino['class'] == 107, 'class'] = 7
df_treino.loc[df_treino['class'] == 108, 'class'] = 8
df_treino.loc[df_treino['class'] == 109, 'class'] = 9

df_test['class'] = df_test['class'].fillna(0)
df_test.loc[df_test['class'] == 101, 'class'] = 1
df_test.loc[df_test['class'] == 102, 'class'] = 2
df_test.loc[df_test['class'] == 103, 'class'] = 3
df_test.loc[df_test['class'] == 104, 'class'] = 4
df_test.loc[df_test['class'] == 105, 'class'] = 5
df_test.loc[df_test['class'] == 106, 'class'] = 6
df_test.loc[df_test['class'] == 107, 'class'] = 7
df_test.loc[df_test['class'] == 108, 'class'] = 8
df_test.loc[df_test['class'] == 109, 'class'] = 9

In [16]:
import logging
import warnings
from argparse import Namespace
from copy import deepcopy
from math import ceil

import torch
from huggingface_hub import PyTorchModelHubMixin
from torch import nn
from transformers import T5Config, T5EncoderModel, T5Model

In [17]:
class NamespaceWithDefaults(Namespace):
    @classmethod
    def from_namespace(cls, namespace):
        new_instance = cls()
        for attr in dir(namespace):
            if not attr.startswith("__"):
                setattr(new_instance, attr, getattr(namespace, attr))
        return new_instance

    def getattr(self, key, default=None):
        return getattr(self, key, default)


In [18]:
from dataclasses import dataclass
import numpy.typing as npt
@dataclass
class TimeseriesOutputs:
    forecast: npt.NDArray = None
    anomaly_scores: npt.NDArray = None
    logits: npt.NDArray = None
    labels: int = None
    input_mask: npt.NDArray = None
    pretrain_mask: npt.NDArray = None
    reconstruction: npt.NDArray = None
    embeddings: npt.NDArray = None
    metadata: dict = None
    illegal_output: bool = False

In [19]:
@dataclass
class TASKS:
    RECONSTRUCTION: str = "reconstruction"
    FORECASTING: str = "forecasting"
    CLASSIFICATION: str = "classification"
    EMBED: str = "embedding"

In [20]:
class MOMENT(nn.Module):
    def __init__(self, config: Namespace | dict, **kwargs: dict):
        super().__init__()
        config = self._update_inputs(config, **kwargs)
        config = self._validate_inputs(config)
        self.config = config
        self.task_name = config.task_name
        self.seq_len = config.seq_len
        self.patch_len = config.patch_len

        self.normalizer = RevIN(
            num_features=1, affine=config.getattr("revin_affine", False)
        )
        self.tokenizer = Patching(
            patch_len=config.patch_len, stride=config.patch_stride_len
        )
        self.patch_embedding = PatchEmbedding(
            d_model=config.d_model,
            seq_len=config.seq_len,
            patch_len=config.patch_len,
            stride=config.patch_stride_len,
            patch_dropout=config.getattr("patch_dropout", 0.1),
            add_positional_embedding=config.getattr("add_positional_embedding", True),
            value_embedding_bias=config.getattr("value_embedding_bias", False),
            orth_gain=config.getattr("orth_gain", 1.41),
        )
        self.mask_generator = Masking(mask_ratio=config.getattr("mask_ratio", 0.0))
        self.encoder = self._get_transformer_backbone(config)
        self.head = self._get_head(self.task_name)

        # Frozen parameters
        self.freeze_embedder = config.getattr("freeze_embedder", True)
        self.freeze_encoder = config.getattr("freeze_encoder", True)
        self.freeze_head = config.getattr("freeze_head", False)

        if self.freeze_embedder:
            self.patch_embedding = freeze_parameters(self.patch_embedding)
        if self.freeze_encoder:
            self.encoder = freeze_parameters(self.encoder)
        if self.freeze_head:
            self.head = freeze_parameters(self.head)

    def _update_inputs(
        self, config: Namespace | dict, **kwargs: dict
    ) -> NamespaceWithDefaults:
        if isinstance(config, dict) and "model_kwargs" in kwargs:
            return NamespaceWithDefaults(**{**config, **kwargs["model_kwargs"]})
        else:
            return NamespaceWithDefaults.from_namespace(config)

    def _validate_inputs(self, config: NamespaceWithDefaults) -> NamespaceWithDefaults:
        if (
            config.d_model is None
            and config.transformer_backbone in SUPPORTED_HUGGINGFACE_MODELS
        ):
            config.d_model = config.t5_config['d_model']
            logging.info(f"Setting d_model to {config.d_model}")
        elif config.d_model is None:
            raise ValueError(
                "d_model must be specified if transformer backbone "
                "unless transformer backbone is a Huggingface model."
            )

        if config.transformer_type not in [
            "encoder_only",
            "decoder_only",
            "encoder_decoder",
        ]:
            raise ValueError(
                "transformer_type must be one of "
                "['encoder_only', 'decoder_only', 'encoder_decoder']"
            )

        if config.patch_stride_len != config.patch_len:
            warnings.warn("Patch stride length is not equal to patch length.")
        return config

    def _get_head(self, task_name: str) -> nn.Module:
        if task_name != TASKS.RECONSTRUCTION:
            warnings.warn("Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.")
        if task_name == TASKS.RECONSTRUCTION:
            return PretrainHead(
                self.config.d_model,
                self.config.patch_len,
                self.config.getattr("head_dropout", 0.1),
                self.config.getattr("orth_gain", 1.41),
            )
        elif task_name == TASKS.CLASSIFICATION:
            return ClassificationHead(
                self.config.n_channels,
                self.config.d_model,
                self.config.num_class,
                self.config.getattr("head_dropout", 0.1),
                reduction = self.config.getattr("reduction", "concat"),
            )
        elif task_name == TASKS.FORECASTING:
            num_patches = (
                max(self.config.seq_len, self.config.patch_len) - self.config.patch_len
            ) // self.config.patch_stride_len + 1
            self.head_nf = self.config.d_model * num_patches
            return ForecastingHead(
                self.head_nf,
                self.config.forecast_horizon,
                self.config.getattr("head_dropout", 0.1),
            )
        elif task_name == TASKS.EMBED:
            return nn.Identity()
        else:
            raise NotImplementedError(f"Task {task_name} not implemented.")

    def _get_transformer_backbone(self, config) -> nn.Module:
        model_config = T5Config.from_dict(config.t5_config)
        if config.getattr("randomly_initialize_backbone", False):
            transformer_backbone = T5Model(model_config)
            logging.info(
                f"Initializing randomly initialized transformer from {config.transformer_backbone}."
            )
        else:
            transformer_backbone = T5EncoderModel(model_config)
            logging.info(
                f"Initializing pre-trained transformer from {config.transformer_backbone}."
            )

        transformer_backbone = transformer_backbone.get_encoder()

        if config.getattr("enable_gradient_checkpointing", True):
            transformer_backbone.gradient_checkpointing_enable()
            logging.info("Enabling gradient checkpointing.")

        return transformer_backbone

    def __call__(self, *args, **kwargs) -> TimeseriesOutputs:
        return self.forward(*args, **kwargs)

    def embed(
        self,
        *,
        x_enc: torch.Tensor,
        input_mask: torch.Tensor = None,
        reduction: str = "mean",
        **kwargs,
    ) -> TimeseriesOutputs:
        """
        Embeds the input time series data into a latent representation using the model's encoder.
    
        Args:
            x_enc (torch.Tensor): The input tensor of shape (batch_size, n_channels, seq_len).
            input_mask (torch.Tensor, optional): A mask tensor of shape (batch_size, seq_len) indicating the valid input positions. Default is None, which means all positions are valid.
            reduction (str, optional): The reduction method to apply on the output embeddings. Options are "mean" (default) or "none". The `mean` reduction averages embeddings over channels and patches. 
            **kwargs: Additional keyword arguments.
    
        Returns:
            TimeseriesOutputs: An object containing the embeddings, input mask, and metadata regarding the reduction method used.
        """
        batch_size, n_channels, seq_len = x_enc.shape

        if input_mask is None:
            input_mask = torch.ones((batch_size, seq_len)).to(x_enc.device)

        x_enc = self.normalizer(x=x_enc, mask=input_mask, mode="norm")
        x_enc = torch.nan_to_num(x_enc, nan=0, posinf=0, neginf=0)

        input_mask_patch_view = Masking.convert_seq_to_patch_view(
            input_mask, self.patch_len
        )

        x_enc = self.tokenizer(x=x_enc)
        enc_in = self.patch_embedding(x_enc, mask=input_mask)

        n_patches = enc_in.shape[2]
        enc_in = enc_in.reshape(
            (batch_size * n_channels, n_patches, self.config.d_model)
        )

        patch_view_mask = Masking.convert_seq_to_patch_view(input_mask, self.patch_len)
        attention_mask = patch_view_mask.repeat_interleave(n_channels, dim=0)
        outputs = self.encoder(inputs_embeds=enc_in, attention_mask=attention_mask)
        enc_out = outputs.last_hidden_state

        enc_out = enc_out.reshape((-1, n_channels, n_patches, self.config.d_model))
        # [batch_size x n_channels x n_patches x d_model]

        if reduction == "mean":
            enc_out = enc_out.mean(dim=1, keepdim=False)  # Mean across channels
            # [batch_size x n_patches x d_model]
            input_mask_patch_view = input_mask_patch_view.unsqueeze(-1).repeat(
                1, 1, self.config.d_model
            )
            enc_out = (input_mask_patch_view * enc_out).sum(
                dim=1
            ) / input_mask_patch_view.sum(dim=1)

        elif reduction == "none":
            pass
        else:
            raise NotImplementedError(f"Reduction method {reduction} not implemented.")

        return TimeseriesOutputs(
            embeddings=enc_out, input_mask=input_mask, metadata=reduction
        )

    def reconstruction(
        self,
        *,
        x_enc: torch.Tensor,
        input_mask: torch.Tensor = None,
        mask: torch.Tensor = None,
        **kwargs,
    ) -> TimeseriesOutputs:
        batch_size, n_channels, _ = x_enc.shape

        if mask is None:
            mask = self.mask_generator.generate_mask(x=x_enc, input_mask=input_mask)
            mask = mask.to(x_enc.device)  # mask: [batch_size x seq_len]

        x_enc = self.normalizer(x=x_enc, mask=mask * input_mask, mode="norm")
        # Prevent too short time-series from causing NaNs
        x_enc = torch.nan_to_num(x_enc, nan=0, posinf=0, neginf=0)

        x_enc = self.tokenizer(x=x_enc)
        enc_in = self.patch_embedding(x_enc, mask=mask)

        n_patches = enc_in.shape[2]
        enc_in = enc_in.reshape(
            (batch_size * n_channels, n_patches, self.config.d_model)
        )

        patch_view_mask = Masking.convert_seq_to_patch_view(input_mask, self.patch_len)
        attention_mask = patch_view_mask.repeat_interleave(n_channels, dim=0)
        if self.config.transformer_type == "encoder_decoder":
            outputs = self.encoder(
                inputs_embeds=enc_in,
                decoder_inputs_embeds=enc_in,
                attention_mask=attention_mask,
            )
        else:
            outputs = self.encoder(inputs_embeds=enc_in, attention_mask=attention_mask)
        enc_out = outputs.last_hidden_state

        enc_out = enc_out.reshape((-1, n_channels, n_patches, self.config.d_model))

        dec_out = self.head(enc_out)  # [batch_size x n_channels x seq_len]
        dec_out = self.normalizer(x=dec_out, mode="denorm")

        if self.config.getattr("debug", False):
            illegal_output = self._check_model_weights_for_illegal_values()
        else:
            illegal_output = None

        return TimeseriesOutputs(
            input_mask=input_mask,
            reconstruction=dec_out,
            pretrain_mask=mask,
            illegal_output=illegal_output,
        )

    def reconstruct(
        self,
        *,
        x_enc: torch.Tensor,
        input_mask: torch.Tensor = None,
        mask: torch.Tensor = None,
        **kwargs,
    ) -> TimeseriesOutputs:
        if mask is None:
            mask = torch.ones_like(input_mask)

        batch_size, n_channels, _ = x_enc.shape
        x_enc = self.normalizer(x=x_enc, mask=mask * input_mask, mode="norm")

        x_enc = self.tokenizer(x=x_enc)
        enc_in = self.patch_embedding(x_enc, mask=mask)

        n_patches = enc_in.shape[2]
        enc_in = enc_in.reshape(
            (batch_size * n_channels, n_patches, self.config.d_model)
        )
        # [batch_size * n_channels x n_patches x d_model]

        patch_view_mask = Masking.convert_seq_to_patch_view(input_mask, self.patch_len)
        attention_mask = patch_view_mask.repeat_interleave(n_channels, dim=0).to(
            x_enc.device
        )

        n_tokens = 0
        if "prompt_embeds" in kwargs:
            prompt_embeds = kwargs["prompt_embeds"].to(x_enc.device)

            if isinstance(prompt_embeds, nn.Embedding):
                prompt_embeds = prompt_embeds.weight.data.unsqueeze(0)

            n_tokens = prompt_embeds.shape[1]

            enc_in = self._cat_learned_embedding_to_input(prompt_embeds, enc_in)
            attention_mask = self._extend_attention_mask(attention_mask, n_tokens)

        if self.config.transformer_type == "encoder_decoder":
            outputs = self.encoder(
                inputs_embeds=enc_in,
                decoder_inputs_embeds=enc_in,
                attention_mask=attention_mask,
            )
        else:
            outputs = self.encoder(inputs_embeds=enc_in, attention_mask=attention_mask)
        enc_out = outputs.last_hidden_state
        enc_out = enc_out[:, n_tokens:, :]

        enc_out = enc_out.reshape((-1, n_channels, n_patches, self.config.d_model))
        # [batch_size x n_channels x n_patches x d_model]

        dec_out = self.head(enc_out)  # [batch_size x n_channels x seq_len]
        dec_out = self.normalizer(x=dec_out, mode="denorm")

        return TimeseriesOutputs(input_mask=input_mask, reconstruction=dec_out)

    def detect_anomalies(
        self,
        *,
        x_enc: torch.Tensor,
        input_mask: torch.Tensor = None,
        anomaly_criterion: str = "mse",
        **kwargs,
    ) -> TimeseriesOutputs:
        outputs = self.reconstruct(x_enc=x_enc, input_mask=input_mask)
        self.anomaly_criterion = get_anomaly_criterion(anomaly_criterion)

        anomaly_scores = self.anomaly_criterion(x_enc, outputs.reconstruction)

        return TimeseriesOutputs(
            input_mask=input_mask,
            reconstruction=outputs.reconstruction,
            anomaly_scores=anomaly_scores,
            metadata={"anomaly_criterion": anomaly_criterion},
        )

    def forecast(
        self,
        *,
        x_enc: torch.Tensor, 
        input_mask: torch.Tensor = None, 
        **kwargs
    ) -> TimeseriesOutputs:
        batch_size, n_channels, seq_len = x_enc.shape

        x_enc = self.normalizer(x=x_enc, mask=input_mask, mode="norm")
        x_enc = torch.nan_to_num(x_enc, nan=0, posinf=0, neginf=0)

        x_enc = self.tokenizer(x=x_enc)
        enc_in = self.patch_embedding(x_enc, mask=torch.ones_like(input_mask))

        n_patches = enc_in.shape[2]
        enc_in = enc_in.reshape(
            (batch_size * n_channels, n_patches, self.config.d_model)
        )

        patch_view_mask = Masking.convert_seq_to_patch_view(input_mask, self.patch_len)
        attention_mask = patch_view_mask.repeat_interleave(n_channels, dim=0)
        outputs = self.encoder(inputs_embeds=enc_in, attention_mask=attention_mask)
        enc_out = outputs.last_hidden_state
        enc_out = enc_out.reshape((-1, n_channels, n_patches, self.config.d_model))
        # [batch_size x n_channels x n_patches x d_model]

        dec_out = self.head(enc_out)  # [batch_size x n_channels x forecast_horizon]
        dec_out = self.normalizer(x=dec_out, mode="denorm")

        return TimeseriesOutputs(input_mask=input_mask, forecast=dec_out)

    def short_forecast(
        self,
        *, 
        x_enc: torch.Tensor,
        input_mask: torch.Tensor = None,
        forecast_horizon: int = 1,
        **kwargs,
    ) -> TimeseriesOutputs:
        batch_size, n_channels, seq_len = x_enc.shape
        num_masked_patches = ceil(forecast_horizon / self.patch_len)
        num_masked_timesteps = num_masked_patches * self.patch_len

        x_enc = self.normalizer(x=x_enc, mask=input_mask, mode="norm")
        x_enc = torch.nan_to_num(x_enc, nan=0, posinf=0, neginf=0)

        # Shift the time-series and mask the last few timesteps for forecasting
        x_enc = torch.roll(x_enc, shifts=-num_masked_timesteps, dims=2)
        input_mask = torch.roll(input_mask, shifts=-num_masked_timesteps, dims=1)

        # Attending to mask tokens
        input_mask[:, -num_masked_timesteps:] = 1
        mask = torch.ones_like(input_mask)
        mask[:, -num_masked_timesteps:] = 0

        x_enc = self.tokenizer(x=x_enc)
        enc_in = self.patch_embedding(x_enc, mask=mask)

        n_patches = enc_in.shape[2]
        enc_in = enc_in.reshape(
            (batch_size * n_channels, n_patches, self.config.d_model)
        )
        # [batch_size * n_channels x n_patches x d_model]

        patch_view_mask = Masking.convert_seq_to_patch_view(input_mask, self.patch_len)
        attention_mask = patch_view_mask.repeat_interleave(n_channels, dim=0)
        outputs = self.encoder(inputs_embeds=enc_in, attention_mask=attention_mask)
        enc_out = outputs.last_hidden_state
        enc_out = enc_out.reshape((-1, n_channels, n_patches, self.config.d_model))

        dec_out = self.head(enc_out)  # [batch_size x n_channels x seq_len]

        end = -num_masked_timesteps + forecast_horizon
        end = None if end == 0 else end

        dec_out = self.normalizer(x=dec_out, mode="denorm")
        forecast = dec_out[:, :, -num_masked_timesteps:end]

        return TimeseriesOutputs(
            input_mask=input_mask,
            reconstruction=dec_out,
            forecast=forecast,
            metadata={"forecast_horizon": forecast_horizon},
        )

    def classify(
        self,
        *,
        x_enc: torch.Tensor,
        input_mask: torch.Tensor = None,
        reduction: str = "concat",
        **kwargs,
    ) -> TimeseriesOutputs:
        batch_size, n_channels, seq_len = x_enc.shape

        if input_mask is None:
            input_mask = torch.ones((batch_size, seq_len)).to(x_enc.device)

        x_enc = self.normalizer(x=x_enc, mask=input_mask, mode="norm")
        x_enc = torch.nan_to_num(x_enc, nan=0, posinf=0, neginf=0)

        input_mask_patch_view = Masking.convert_seq_to_patch_view(
            input_mask, self.patch_len
        )

        x_enc = self.tokenizer(x=x_enc)
        enc_in = self.patch_embedding(x_enc, mask=input_mask)

        n_patches = enc_in.shape[2]
        enc_in = enc_in.reshape(
            (batch_size * n_channels, n_patches, self.config.d_model)
        )

        patch_view_mask = Masking.convert_seq_to_patch_view(input_mask, self.patch_len)
        attention_mask = patch_view_mask.repeat_interleave(n_channels, dim=0)
        outputs = self.encoder(inputs_embeds=enc_in, attention_mask=attention_mask)
        enc_out = outputs.last_hidden_state

        enc_out = enc_out.reshape((-1, n_channels, n_patches, self.config.d_model))
        # [batch_size x n_channels x n_patches x d_model]

        # Mean across channels
        if reduction == "mean":
            # [batch_size x n_patches x d_model]
            enc_out = enc_out.mean(dim=1, keepdim=False)  
        # Concatenate across channels
        elif reduction == "concat":
            # [batch_size x n_patches x d_model * n_channels]
            enc_out = enc_out.permute(0, 2, 3, 1).reshape(
                batch_size, n_patches, self.config.d_model * n_channels)

        else:
            raise NotImplementedError(f"Reduction method {reduction} not implemented.")

        logits = self.head(enc_out, input_mask=input_mask)

        return TimeseriesOutputs(embeddings=enc_out, logits=logits, metadata=reduction)

    def forward(
        self,
        *,
        x_enc: torch.Tensor,
        input_mask: torch.Tensor = None,
        mask: torch.Tensor = None,
        **kwargs,
    ) -> TimeseriesOutputs:
        if input_mask is None:
            input_mask = torch.ones_like(x_enc[:, 0, :])

        if self.task_name == TASKS.RECONSTRUCTION:
            return self.reconstruction(
                x_enc=x_enc, mask=mask, input_mask=input_mask, **kwargs
            )
        elif self.task_name == TASKS.EMBED:
            return self.embed(x_enc=x_enc, input_mask=input_mask, **kwargs)
        elif self.task_name == TASKS.FORECASTING:
            return self.forecast(x_enc=x_enc, input_mask=input_mask, **kwargs)
        elif self.task_name == TASKS.CLASSIFICATION:
            return self.classify(x_enc=x_enc, input_mask=input_mask, **kwargs)
        else:
            raise NotImplementedError(f"Task {self.task_name} not implemented.")

In [21]:
class MOMENTPipeline(MOMENT, PyTorchModelHubMixin):
    def __init__(self, config: Namespace | dict, **kwargs: dict):
        self._validate_model_kwargs(**kwargs)
        self.new_task_name = kwargs.get("model_kwargs", {}).pop(
            "task_name", TASKS.RECONSTRUCTION
        )
        super().__init__(config, **kwargs)

    def _validate_model_kwargs(self, **kwargs: dict) -> None:
        kwargs = deepcopy(kwargs)
        kwargs.setdefault("model_kwargs", {"task_name": TASKS.RECONSTRUCTION})
        kwargs["model_kwargs"].setdefault("task_name", TASKS.RECONSTRUCTION)
        config = Namespace(**kwargs["model_kwargs"])

        if config.task_name == TASKS.FORECASTING:
            if not hasattr(config, "forecast_horizon"):
                raise ValueError(
                    "forecast_horizon must be specified for long-horizon forecasting."
                )

        if config.task_name == TASKS.CLASSIFICATION:
            if not hasattr(config, "n_channels"):
                raise ValueError("n_channels must be specified for classification.")
            if not hasattr(config, "num_class"):
                raise ValueError("num_class must be specified for classification.")

    def init(self) -> None:
        if self.new_task_name != TASKS.RECONSTRUCTION:
            self.task_name = self.new_task_name
            self.head = self._get_head(self.new_task_name)


class ClassificationHead(nn.Module):
    def __init__(
        self,
        n_channels: int = 1,
        d_model: int = 768,
        n_classes: int = 2,
        head_dropout: int = 0.1,
        reduction: str = "concat",
    ):
        super().__init__()
        self.dropout = nn.Dropout(head_dropout)
        if reduction == "mean":
            self.linear = nn.Linear(d_model, n_classes)
        elif reduction == "concat":
            self.linear = nn.Linear(n_channels * d_model, n_classes)
        else:
            raise ValueError(f"Reduction method {reduction} not implemented. Only 'mean' and 'concat' are supported.")

    def forward(self, x, input_mask: torch.Tensor = None):
        x = torch.mean(x, dim=1)
        x = self.dropout(x)
        y = self.linear(x)
        return y

In [22]:
SUPPORTED_HUGGINGFACE_MODELS = [
    "google/flan-t5-small",
    "google/flan-t5-base",
    "google/flan-t5-large",
    "google/flan-t5-xl",
    "google/flan-t5-xxl",
]

In [23]:
class PretrainHead(nn.Module):
    def __init__(
        self,
        d_model: int = 768,
        patch_len: int = 8,
        head_dropout: float = 0.1,
        orth_gain: float = 1.41,
    ):
        super().__init__()
        self.dropout = nn.Dropout(head_dropout)
        self.linear = nn.Linear(d_model, patch_len)

        if orth_gain is not None:
            torch.nn.init.orthogonal_(self.linear.weight, gain=orth_gain)
            self.linear.bias.data.zero_()

    def forward(self, x):
        x = self.linear(self.dropout(x))
        x = x.flatten(start_dim=2, end_dim=3)
        return x


In [24]:
def nanvar(tensor, dim=None, keepdim=False):
    tensor_mean = tensor.nanmean(dim=dim, keepdim=True)
    output = (tensor - tensor_mean).square().nanmean(dim=dim, keepdim=keepdim)
    return output


def nanstd(tensor, dim=None, keepdim=False):
    output = nanvar(tensor, dim=dim, keepdim=keepdim)
    output = output.sqrt()
    return output


class RevIN(nn.Module):
    def __init__(self, num_features: int, eps: float = 1e-5, affine: bool = False):
        """
        :param num_features: the number of features or channels
        :param eps: a value added for numerical stability
        :param affine: if True, RevIN has learnable affine parameters
        """
        super(RevIN, self).__init__()
        self.num_features = num_features
        self.eps = eps
        self.affine = affine

        if self.affine:
            self._init_params()

    def forward(self, x: torch.Tensor, mode: str = "norm", mask: torch.Tensor = None):
        """
        :param x: input tensor of shape (batch_size, n_channels, seq_len)
        :param mode: 'norm' or 'denorm'
        :param mask: input mask of shape (batch_size, seq_len)
        :return: RevIN transformed tensor
        """
        if mode == "norm":
            self._get_statistics(x, mask=mask)
            x = self._normalize(x)
        elif mode == "denorm":
            x = self._denormalize(x)
        else:
            raise NotImplementedError
        return x

    def _init_params(self):
        # initialize RevIN params: (C,)
        self.affine_weight = nn.Parameter(torch.ones(1, self.num_features, 1))
        self.affine_bias = nn.Parameter(torch.zeros(1, self.num_features, 1))

    def _get_statistics(self, x, mask=None):
        """
        x    : batch_size x n_channels x seq_len
        mask : batch_size x seq_len
        """
        if mask is None:
            mask = torch.ones((x.shape[0], x.shape[-1]))
        n_channels = x.shape[1]
        mask = mask.unsqueeze(1).repeat(1, n_channels, 1).bool()
        # Set masked positions to NaN, and unmasked positions are taken from x
        masked_x = torch.where(mask, x, torch.nan)
        self.mean = torch.nanmean(masked_x, dim=-1, keepdim=True).detach()
        self.stdev = nanstd(masked_x, dim=-1, keepdim=True).detach() + self.eps
        # self.stdev = torch.sqrt(
        #     torch.var(masked_x, dim=-1, keepdim=True) + self.eps).get_data().detach()
        # NOTE: By default not bessel correction

    def _normalize(self, x):
        x = x - self.mean
        x = x / self.stdev

        if self.affine:
            x = x * self.affine_weight
            x = x + self.affine_bias
        return x

    def _denormalize(self, x):
        if self.affine:
            x = x - self.affine_bias
            x = x / (self.affine_weight + self.eps * self.eps)
        x = x * self.stdev
        x = x + self.mean
        return x

In [25]:
import math
import warnings

import torch
import torch.nn as nn



class PositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_len=5000, model_name="MOMENT"):
        super(PositionalEmbedding, self).__init__()
        self.model_name = model_name

        # Compute the positional encodings once in log space.
        pe = torch.zeros(max_len, d_model).float()
        pe.require_grad = False

        position = torch.arange(0, max_len).float().unsqueeze(1)
        div_term = (
            torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model)
        ).exp()

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        if (
            self.model_name == "MOMENT"
            or self.model_name == "TimesNet"
            or self.model_name == "GPT4TS"
        ):
            return self.pe[:, : x.size(2)]
        else:
            return self.pe[:, : x.size(1)]


class TokenEmbedding(nn.Module):
    def __init__(self, c_in, d_model):
        super(TokenEmbedding, self).__init__()
        padding = 1 if torch.__version__ >= "1.5.0" else 2
        self.tokenConv = nn.Conv1d(
            in_channels=c_in,
            out_channels=d_model,
            kernel_size=3,
            padding=padding,
            padding_mode="circular",
            bias=False,
        )
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(
                    m.weight, mode="fan_in", nonlinearity="leaky_relu"
                )

    def forward(self, x):
        # x = x.permute(0, 2, 1)
        x = self.tokenConv(x)
        x = x.transpose(1, 2)
        # batch_size x seq_len x d_model
        return x


class FixedEmbedding(nn.Module):
    def __init__(self, c_in, d_model):
        super(FixedEmbedding, self).__init__()

        w = torch.zeros(c_in, d_model).float()
        w.require_grad = False

        position = torch.arange(0, c_in).float().unsqueeze(1)
        div_term = (
            torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model)
        ).exp()

        w[:, 0::2] = torch.sin(position * div_term)
        w[:, 1::2] = torch.cos(position * div_term)

        self.emb = nn.Embedding(c_in, d_model)
        self.emb.weight = nn.Parameter(w, requires_grad=False)

    def forward(self, x):
        return self.emb(x).detach()


class TemporalEmbedding(nn.Module):
    def __init__(self, d_model, embed_type="fixed", freq="h"):
        super(TemporalEmbedding, self).__init__()

        minute_size = 4
        hour_size = 24
        weekday_size = 7
        day_size = 32
        month_size = 13

        Embed = FixedEmbedding if embed_type == "fixed" else nn.Embedding
        if freq == "t":
            self.minute_embed = Embed(minute_size, d_model)
        self.hour_embed = Embed(hour_size, d_model)
        self.weekday_embed = Embed(weekday_size, d_model)
        self.day_embed = Embed(day_size, d_model)
        self.month_embed = Embed(month_size, d_model)

    def forward(self, x):
        x = x.long()
        minute_x = (
            self.minute_embed(x[:, :, 4]) if hasattr(self, "minute_embed") else 0.0
        )
        hour_x = self.hour_embed(x[:, :, 3])
        weekday_x = self.weekday_embed(x[:, :, 2])
        day_x = self.day_embed(x[:, :, 1])
        month_x = self.month_embed(x[:, :, 0])

        return hour_x + weekday_x + day_x + month_x + minute_x


class TimeFeatureEmbedding(nn.Module):
    def __init__(self, d_model, embed_type="timeF", freq="h"):
        super(TimeFeatureEmbedding, self).__init__()

        freq_map = {"h": 4, "t": 5, "s": 6, "m": 1, "a": 1, "w": 2, "d": 3, "b": 3}
        d_inp = freq_map[freq]
        self.embed = nn.Linear(d_inp, d_model, bias=False)

    def forward(self, x):
        return self.embed(x)


class DataEmbedding(nn.Module):
    def __init__(
        self, c_in, d_model, model_name, embed_type="fixed", freq="h", patch_dropout=0.1
    ):
        super(DataEmbedding, self).__init__()
        self.value_embedding = TokenEmbedding(c_in=c_in, d_model=d_model)
        self.position_embedding = PositionalEmbedding(
            d_model=d_model, model_name=model_name
        )
        self.temporal_embedding = (
            TemporalEmbedding(d_model=d_model, embed_type=embed_type, freq=freq)
            if embed_type != "timeF"
            else TimeFeatureEmbedding(d_model=d_model, embed_type=embed_type, freq=freq)
        )
        self.dropout = nn.Dropout(p=patch_dropout)

    def forward(self, x, x_mark=None):
        if x_mark is None:
            x = self.value_embedding(x) + self.position_embedding(x)
        else:
            x = (
                self.value_embedding(x)
                + self.temporal_embedding(x_mark)
                + self.position_embedding(x)
            )
        return self.dropout(x)


class DataEmbedding_wo_pos(nn.Module):
    def __init__(self, c_in, d_model, embed_type="fixed", freq="h", patch_dropout=0.1):
        super(DataEmbedding_wo_pos, self).__init__()

        self.value_embedding = TokenEmbedding(c_in=c_in, d_model=d_model)
        self.position_embedding = PositionalEmbedding(d_model=d_model)
        self.temporal_embedding = (
            TemporalEmbedding(d_model=d_model, embed_type=embed_type, freq=freq)
            if embed_type != "timeF"
            else TimeFeatureEmbedding(d_model=d_model, embed_type=embed_type, freq=freq)
        )
        self.dropout = nn.Dropout(p=patch_dropout)

    def forward(self, x, x_mark):
        if x_mark is None:
            x = self.value_embedding(x)
        else:
            x = self.value_embedding(x) + self.temporal_embedding(x_mark)
        return self.dropout(x)


class PatchEmbedding(nn.Module):
    def __init__(
        self,
        d_model: int = 768,
        seq_len: int = 512,
        patch_len: int = 8,
        stride: int = 8,
        patch_dropout: int = 0.1,
        add_positional_embedding: bool = False,
        value_embedding_bias: bool = False,
        orth_gain: float = 1.41,
    ):
        super(PatchEmbedding, self).__init__()
        self.patch_len = patch_len
        self.seq_len = seq_len
        self.stride = stride
        self.d_model = d_model
        self.add_positional_embedding = add_positional_embedding

        self.value_embedding = nn.Linear(patch_len, d_model, bias=value_embedding_bias)
        self.mask_embedding = nn.Parameter(torch.zeros(d_model))

        if orth_gain is not None:
            torch.nn.init.orthogonal_(self.value_embedding.weight, gain=orth_gain)
            if value_embedding_bias:
                self.value_embedding.bias.data.zero_()
            # torch.nn.init.orthogonal_(self.mask_embedding, gain=orth_gain) # Fails

        # Positional embedding
        if self.add_positional_embedding:
            self.position_embedding = PositionalEmbedding(d_model)

        # Residual dropout
        self.dropout = nn.Dropout(patch_dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        mask = Masking.convert_seq_to_patch_view(
            mask, patch_len=self.patch_len
        ).unsqueeze(-1)
        # mask : [batch_size x n_patches x 1]
        n_channels = x.shape[1]
        mask = (
            mask.repeat_interleave(self.d_model, dim=-1)
            .unsqueeze(1)
            .repeat(1, n_channels, 1, 1)
        )
        # mask : [batch_size x n_channels x n_patches x d_model]

        # Input encoding
        x = mask * self.value_embedding(x) + (1 - mask) * self.mask_embedding
        if self.add_positional_embedding:
            x = x + self.position_embedding(x)

        return self.dropout(x)


class Patching(nn.Module):
    def __init__(self, patch_len: int, stride: int):
        super().__init__()
        self.patch_len = patch_len
        self.stride = stride
        if self.stride != self.patch_len:
            warnings.warn(
                "Stride and patch length are not equal. "
                "This may lead to unexpected behavior."
            )

    def forward(self, x):
        x = x.unfold(dimension=-1, size=self.patch_len, step=self.stride)
        # x : [batch_size x n_channels x num_patch x patch_len]
        return x

In [26]:
from typing import Optional

import torch


class Masking:
    def __init__(
        self, mask_ratio: float = 0.3, patch_len: int = 8, stride: Optional[int] = None
    ):
        """
        Indices with 0 mask are hidden, and with 1 are observed.
        """
        self.mask_ratio = mask_ratio
        self.patch_len = patch_len
        self.stride = patch_len if stride is None else stride

    @staticmethod
    def convert_seq_to_patch_view(
        mask: torch.Tensor, patch_len: int = 8, stride: Optional[int] = None
    ):
        """
        Input:
            mask : torch.Tensor of shape [batch_size x seq_len]
        Output
            mask : torch.Tensor of shape [batch_size x n_patches]
        """
        stride = patch_len if stride is None else stride
        mask = mask.unfold(dimension=-1, size=patch_len, step=stride)
        # mask : [batch_size x n_patches x patch_len]
        return (mask.sum(dim=-1) == patch_len).long()

    @staticmethod
    def convert_patch_to_seq_view(
        mask: torch.Tensor,
        patch_len: int = 8,
    ):
        """
        Input:
            mask : torch.Tensor of shape [batch_size x n_patches]
        Output:
            mask : torch.Tensor of shape [batch_size x seq_len]
        """
        return mask.repeat_interleave(patch_len, dim=-1)

    def generate_mask(self, x: torch.Tensor, input_mask: Optional[torch.Tensor] = None):
        """
        Input:
            x : torch.Tensor of shape
            [batch_size x n_channels x n_patches x patch_len] or
            [batch_size x n_channels x seq_len]
            input_mask: torch.Tensor of shape [batch_size x seq_len] or
            [batch_size x n_patches]
        Output:
            mask : torch.Tensor of shape [batch_size x seq_len]
        """
        if x.ndim == 4:
            return self._mask_patch_view(x, input_mask=input_mask)
        elif x.ndim == 3:
            return self._mask_seq_view(x, input_mask=input_mask)

    def _mask_patch_view(self, x, input_mask=None):
        """
        Input:
            x : torch.Tensor of shape
            [batch_size x n_channels x n_patches x patch_len]
            input_mask: torch.Tensor of shape [batch_size x seq_len]
        Output:
            mask : torch.Tensor of shape [batch_size x n_patches]
        """
        input_mask = self.convert_seq_to_patch_view(
            input_mask, self.patch_len, self.stride
        )
        n_observed_patches = input_mask.sum(dim=-1, keepdim=True)  # batch_size x 1

        batch_size, _, n_patches, _ = x.shape
        len_keep = torch.ceil(n_observed_patches * (1 - self.mask_ratio)).long()
        noise = torch.rand(
            batch_size, n_patches, device=x.device
        )  # noise in [0, 1], batch_size x n_channels x n_patches
        noise = torch.where(
            input_mask == 1, noise, torch.ones_like(noise)
        )  # only keep the noise of observed patches

        # Sort noise for each sample
        ids_shuffle = torch.argsort(
            noise, dim=1
        )  # Ascend: small is keep, large is remove
        ids_restore = torch.argsort(
            ids_shuffle, dim=1
        )  # ids_restore: [batch_size x n_patches]

        # Generate the binary mask: 0 is keep, 1 is remove
        mask = torch.zeros(
            [batch_size, n_patches], device=x.device
        )  # mask: [batch_size x n_patches]
        for i in range(batch_size):
            mask[i, : len_keep[i]] = 1

        # Unshuffle to get the binary mask
        mask = torch.gather(mask, dim=1, index=ids_restore)

        return mask.long()

    def _mask_seq_view(self, x, input_mask=None):
        """
        Input:
            x : torch.Tensor of shape
            [batch_size x n_channels x seq_len]
            input_mask: torch.Tensor of shape [batch_size x seq_len]
        Output:
            mask : torch.Tensor of shape [batch_size x seq_len]
        """
        x = x.unfold(dimension=-1, size=self.patch_len, step=self.stride)
        mask = self._mask_patch_view(x, input_mask=input_mask)
        return self.convert_patch_to_seq_view(mask, self.patch_len).long()

In [27]:
def freeze_parameters(model):
    """
    Freeze parameters of the model
    """
    # Freeze the parameters
    for name, param in model.named_parameters():
        param.requires_grad = False

    return model

In [28]:
df_treino.drop([ 'Diretorio'], axis=1,inplace=True)
df_test.drop([ 'Diretorio'], axis=1,inplace=True)

In [29]:
label_col = "class"  # Troque para o nome correto se for diferente
# Descobrir número de canais (atributos)
feature_cols = [col for col in df_treino.columns if col != label_col]

In [30]:
import numpy as np
import pandas as pd

window_size = 128  # ajuste conforme necessário

FEATURE_COLS = [
    'P-PDG',
    'P-TPT',
    'T-TPT',
    'P-MON-CKP',
    'T-JUS-CKP',
    'P-JUS-CKGL',
    'T-JUS-CKGL',
    'QGL',
]
def create_windows_np(group):
    arr = group[FEATURE_COLS].values          # shape: (N, 8)
    labels = group['class'].values
    if len(arr) < window_size:
        return pd.DataFrame()
  # Cria janelas (window_size, 8)
    windows = [
        arr[i - window_size : i]
        for i in range(window_size, len(arr) + 1)
    ]
    window_labels = labels[window_size - 1 :]
    return pd.DataFrame({
        'window': windows,
        'class': window_labels
    })

# Ordene o dataframe antes!
df = df_test[['instance_id', 'timestamp', 'P-PDG','P-TPT','T-TPT','P-MON-CKP','T-JUS-CKP','P-JUS-CKGL','T-JUS-CKGL','QGL', 'class']]
df = df.sort_values(['instance_id', 'timestamp'])

# Aplique para cada instance_id
windows_test = df.groupby('instance_id', group_keys=False).apply(create_windows_np).reset_index(drop=True)

X = np.vstack(windows_test['window'].values)
y = windows_test['class'].values
# Crie um DataFrame onde cada coluna é um ponto da série temporal
df_windows_test = pd.DataFrame(X)
df_windows_test['class'] = y


ValueError: Length of values (140004) does not match length of index (17920512)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# Descobrir nome da coluna de rótulo (ajuste se necessário)
label_col = "class"  # Troque para o nome correto se for diferente
# Descobrir número de canais (atributos)
df.drop(['instance_id','timestamp'], axis=1,inplace=True)
feature_cols = [col for col in df.columns if col != label_col]
n_channels = len(feature_cols)
# Descobrir número de classes
num_class = df[label_col].nunique()
# Carregar modelo
model = MOMENTPipeline.from_pretrained(
    "./MOMENT-1-large",
    model_kwargs={
        'task_name': 'classification',
        'n_channels': n_channels,
        'num_class': num_class
    },
    local_files_only=True
)



In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

class ClassificationDataset:
    def __init__(self, data_split="train", dataframe=None, label_col=None, seq_len=128):
        """
        Parâmetros
        ----------
        data_split : str
            Split do dataset, 'train', 'val' ou 'test' (mantido para compatibilidade).
        dataframe : pd.DataFrame, opcional
            DataFrame contendo os dados de séries temporais e as labels.
            As colunas das séries temporais devem ser numéricas e label_col deve indicar a coluna da label.
        label_col : str, opcional
            Nome da coluna no dataframe que contém as labels.
        seq_len : int
            Comprimento da sequência para padding.
        """

        self.seq_len = seq_len
        self.data_split = data_split
        self.dataframe = dataframe
        self.label_col = label_col

        if dataframe is None:
            # Se não for passado dataframe, mantém comportamento original (leitura dos arquivos .ts)
            self.train_file_path_and_name = "../data/ECG5000_TRAIN.ts"
            self.test_file_path_and_name = "../data/ECG5000_TEST.ts"
            self._read_data()
        else:
            # Se dataframe foi passado, processa os dados do dataframe
            self._read_from_dataframe()

    def _transform_labels(self, train_labels: np.ndarray, test_labels: np.ndarray):
        labels = np.unique(train_labels)  # Move labels para {0, ..., L-1}
        transform = {l: i for i, l in enumerate(labels)}

        train_labels = np.vectorize(transform.get)(train_labels)
        test_labels = np.vectorize(transform.get)(test_labels)

        return train_labels, test_labels

    def __len__(self):
        return self.num_timeseries

    def _read_data(self):
        from momentfm.utils.data import load_from_tsfile

        self.scaler = StandardScaler()

        self.train_data, self.train_labels = load_from_tsfile(self.train_file_path_and_name)
        self.test_data, self.test_labels = load_from_tsfile(self.test_file_path_and_name)

        self.train_labels, self.test_labels = self._transform_labels(self.train_labels, self.test_labels)

        if self.data_split == "train":
            self.data = self.train_data
            self.labels = self.train_labels
        else:
            self.data = self.test_data
            self.labels = self.test_labels

        self.num_timeseries = self.data.shape[0]
        self.len_timeseries = self.data.shape[2]

        self.data = self.data.reshape(-1, self.len_timeseries)
        self.scaler.fit(self.data)
        self.data = self.scaler.transform(self.data)
        self.data = self.data.reshape(self.num_timeseries, self.len_timeseries)

        self.data = self.data.T

    def _read_from_dataframe(self):
        """
        Processa o dataframe passado para criar os atributos data e labels.
        Assume que o dataframe tem uma coluna para labels e as demais colunas são dados da série temporal.
        """

        if self.label_col is None:
            raise ValueError("Quando dataframe é passado, label_col deve ser informada.")

        # Separa os labels e os dados
        labels = self.dataframe[self.label_col].values
        data_df = self.dataframe.drop(columns=[self.label_col])

        # Converte dados para numpy array
        data = data_df.values

        # Padroniza os dados (normalização de cada série temporal)
        self.scaler = StandardScaler()
        data = self.scaler.fit_transform(data)

        # Transpõe para manter formato compatível com o original (len_timeseries, num_timeseries)
        self.data = data.T

        self.labels = labels.astype(int)
        self.num_timeseries = self.data.shape[1]
        self.len_timeseries = self.data.shape[0]

    def __getitem__(self, index):
        assert index < self.__len__()
        timeseries = self.data[:, index]
        timeseries_len = len(timeseries)
        labels = self.labels[index].astype(int)
      # Se a série for maior que seq_len, corte o início
        if timeseries_len > self.seq_len:
            timeseries = timeseries[-self.seq_len:]
            timeseries_len = self.seq_len  # Atualize o tamanho
        pad_len = self.seq_len - timeseries_len
        input_mask = np.ones(self.seq_len)
        if pad_len > 0:
            input_mask[:pad_len] = 0
            timeseries = np.pad(timeseries, (pad_len, 0))
        # Se pad_len == 0, não faz nada
        # Se pad_len < 0, nunca entra aqui pois já cortou acima
        return np.expand_dims(timeseries, axis=0), input_mask, labels

# Exemplo de uso:
# df_treino = pd.read_csv("seu_arquivo.csv")

# Exemplo de uso:
train_dataset = ClassificationDataset(dataframe=df_windows_train, label_col="class")
test_dataset = ClassificationDataset(dataframe=df_windows_test, label_col="class")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

idx = np.random.randint(0, len(train_dataset))
heartbeat_start = np.argmax(train_dataset[idx][1])
heartbeat = train_dataset[idx][0].squeeze()[heartbeat_start:]
label = train_dataset[idx][2]
plt.plot(heartbeat, c='darkblue')
plt.title(f"idx={idx} | label={label}")
plt.show()

In [ ]:
from torch.utils.data import DataLoader
train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=False, drop_last=False)
test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False, drop_last=False)

In [ ]:
model.init()

In [ ]:
import torch
import numpy as np
from tqdm import tqdm


def get_embedding(model, dataloader, device):
    embeddings, labels = [], []

    model.to(device)
    model.eval()  # importante colocar o modelo em modo avaliação

    with torch.no_grad():
        for batch_x, batch_masks, batch_labels in tqdm(dataloader, total=len(dataloader)):
            batch_x = batch_x.to(device).float()
            batch_masks = batch_masks.to(device)
            batch_labels = batch_labels.to("cpu").numpy()  # garante que labels estejam em cpu para concatenar

            output = model(x_enc=batch_x, input_mask=batch_masks)  # saída do modelo
            embedding = output.embeddings  # supondo que o output tenha atributo embeddings
            if embedding is not None:
                embeddings.append(embedding.detach().cpu().numpy())
            # else:
                # print("Atenção: embedding veio como None!")
            labels.append(batch_labels)

    embeddings = np.concatenate(embeddings)
    labels = np.concatenate(labels)

    return embeddings, labels



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_embeddings, train_labels = get_embedding(model, train_dataloader,device)
test_embeddings, test_labels = get_embedding(model, test_dataloader,device)

print(train_embeddings.shape, train_labels.shape)
print(test_embeddings.shape, test_labels.shape)


In [ ]:
from sklearn.svm import SVC
def fit_svm(
    features: npt.NDArray,
    y: npt.NDArray,
    MAX_SAMPLES: int = 2000,
):
    # Número de classes
    nb_classes = np.unique(y).shape[0]
    train_size = features.shape[0]
  # Modelo SVM padrão
    svm = SVC(
        C=1,
        kernel="rbf",
        gamma="scale",
        max_iter=10_000_000,
    )
  # Caso de dataset muito pequeno
    if train_size // nb_classes < 5 or train_size < 50:
        return svm.fit(features, y)
  # Subamostragem estratificada se o dataset for grande
    if train_size > MAX_SAMPLES:
        X_sub, _, y_sub, _ = train_test_split(
            features,
            y,
            train_size=MAX_SAMPLES,
            random_state=0,
            stratify=y,
        )
    else:
        X_sub = features
        y_sub = y
  # Treino final
    svm.fit(X_sub, y_sub)
    return svm
def preprocess_embeddings(X: np.ndarray) -> np.ndarray:
    """
    Converte embeddings (N, T, D) para (N, D) usando mean pooling.
    """
    if X.ndim == 3:
        return X.mean(axis=1)
    elif X.ndim == 2:
        return X
    else:
        raise ValueError(f"Formato inesperado de embeddings: {X.shape}")

# Preprocessa embeddings
train_embeddings_2d = preprocess_embeddings(train_embeddings)
test_embeddings_2d = preprocess_embeddings(test_embeddings)
# Treina o classificador
clf = fit_svm(
    features=train_embeddings_2d,
    y=train_labels
)
# Predições
y_pred_train = clf.predict(train_embeddings_2d)
y_pred_test = clf.predict(test_embeddings_2d)
# Avaliação
train_accuracy = clf.score(train_embeddings_2d, train_labels)
test_accuracy = clf.score(test_embeddings_2d, test_labels)
print(f"Train accuracy: {train_accuracy:.2f}")
print(f"Test accuracy: {test_accuracy:.2f}")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
# Matriz de confusão para o conjunto de treino
cm_train = confusion_matrix(train_labels, y_pred_train)
disp_train = ConfusionMatrixDisplay(confusion_matrix=cm_train)
disp_train.plot(cmap=plt.cm.Blues)
plt.title("Matriz de Confusão - Treino")
plt.show()
# Matriz de confusão para o conjunto de teste
cm_test = confusion_matrix(test_labels, y_pred_test)
disp_test = ConfusionMatrixDisplay(confusion_matrix=cm_test)
disp_test.plot(cmap=plt.cm.Blues)
plt.title("Matriz de Confusão - Teste")
plt.show()

In [ ]:
fgf

In [ ]:
import numpy as np
import numpy.typing as npt
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import SVC


def fit_svm(features: npt.NDArray, y: npt.NDArray, MAX_SAMPLES: int = 10000):
    nb_classes = np.unique(y, return_counts=True)[1].shape[0]
    train_size = features.shape[0]

    svm = SVC(C=100000, gamma="scale")
    if train_size // nb_classes < 5 or train_size < 50:
        return svm.fit(features, y)
    else:
        grid_search = GridSearchCV(
            svm,
            {
                "C": [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000],
                "kernel": ["rbf"],
                "degree": [3],
                "gamma": ["scale"],
                "coef0": [0],
                "shrinking": [True],
                "probability": [False],
                "tol": [0.001],
                "cache_size": [200],
                "class_weight": [None],
                "verbose": [False],
                "max_iter": [10000000],
                "decision_function_shape": ["ovr"],
            },
            cv=5,
            n_jobs=10,
        )
        # If the training set is too large, subsample MAX_SAMPLES examples
        if train_size > MAX_SAMPLES:
            split = train_test_split(
                features, y, train_size=MAX_SAMPLES, random_state=0, stratify=y
            )
            features = split[0]
            y = split[2]

        grid_search.fit(features, y)
        return grid_search.best_estimator_
clf = fit_svm(features=train_embeddings, y=train_labels)

y_pred_train = clf.predict(train_embeddings)
y_pred_test = clf.predict(test_embeddings)
train_accuracy = clf.score(train_embeddings, train_labels)
test_accuracy = clf.score(test_embeddings, test_labels)

print(f"Train accuracy: {train_accuracy:.2f}")
print(f"Test accuracy: {test_accuracy:.2f}")
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
# Matriz de confusão para o conjunto de treino
cm_train = confusion_matrix(train_labels, y_pred_train)
disp_train = ConfusionMatrixDisplay(confusion_matrix=cm_train)
disp_train.plot(cmap=plt.cm.Blues)
plt.title("Matriz de Confusão - Treino")
plt.show()
# Matriz de confusão para o conjunto de teste
cm_test = confusion_matrix(test_labels, y_pred_test)
disp_test = ConfusionMatrixDisplay(confusion_matrix=cm_test)
disp_test.plot(cmap=plt.cm.Blues)
plt.title("Matriz de Confusão - Teste")
plt.show()

In [ ]:
# Treinar modelo
model.fit(X_train, y_train)
# Fazer predições
y_pred = model.predict(X_test)
# Matriz de confusão
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap=plt.cm.Blues)
plt.title("Matriz de Confusão")
plt.show()

In [ ]:
feature_cols

In [ ]:
dsd

In [ ]:
type(train_events)

In [ ]:
full_dados = train_events + test_events

In [ ]:
import random
# Sua lista de dicionários
# dados = [...]
amostras = {
    0: {'real': {'treino': 468, 'teste': 129}, 'simulado': {'treino': 0, 'teste': 0}},
    1: {'real': {'treino': 3, 'teste': 2},    'simulado': {'treino': 78, 'teste': 36}},
    2: {'real': {'treino': 15, 'teste': 7},   'simulado': {'treino': 11, 'teste': 5}},
    3: {'real': {'treino': 22, 'teste': 10},  'simulado': {'treino': 55, 'teste': 19}},
    4: {'real': {'treino': 260, 'teste': 84}, 'simulado': {'treino': 0, 'teste': 0}},
    5: {'real': {'treino': 8, 'teste': 4},    'simulado': {'treino': 340, 'teste': 99}},
    6: {'real': {'treino': 4, 'teste': 2},    'simulado': {'treino': 170, 'teste': 45}},
    8: {'real': {'treino': 0, 'teste': 3},    'simulado': {'treino': 56, 'teste': 25}},
}
train_events = []
test_events = []
random.seed(42)  # Para reprodutibilidade
for tipo, origens in amostras.items():
    for origem, splits in origens.items():
        if origem == "real":
            filtro = lambda x: x["event_type"] == tipo and "WELL" in x["file_name"]
        else:  # simulado
            filtro = lambda x: x["event_type"] == tipo and "WELL" not in x["file_name"]
        
        # Filtra os dados
        items_filtrados = [x for x in full_dados if isinstance(x, dict) and filtro(x)]
        # Embaralha para garantir aleatoriedade
        random.shuffle(items_filtrados)
        
        n_treino = min(splits["treino"], len(items_filtrados))
        n_teste = min(splits["teste"], max(0, len(items_filtrados) - n_treino))

        train_events.extend(items_filtrados[:n_treino])
        test_events.extend(items_filtrados[n_treino:n_treino + n_teste])
# treino e teste agora têm as amostras separadas conforme o especificado

In [ ]:
!pip install git+https://github.com/moment-timeseries-foundation-model/moment.git

In [ ]:
type(train_events)

In [ ]:
len(test_events)

In [ ]:
experiment_sampler = sample
model_sampler = lightgbm_sampler

In [ ]:
study = hyperparameter_search(train_events, experiment_sampler, model_sampler, config)

In [ ]:
best_hyperparams = study.best_params
print(best_hyperparams)

In [ ]:
best_params = {'level': 9, 'stride': 10, 'n_components': 0.9465388619191799, 'normal_balance': 5, 'subsample': 0.9500000000000001, 'feature_fraction': 0.45000000000000007, 'lambda_l1': 0.0005675726564192757, 'lambda_l2': 0.02990704067661133, 'num_leaves': 115}

In [ ]:
best_experiment = Experiment(
    level=best_params['level'],
    stride=best_params['stride'],
    n_components=best_params['n_components'],
    normal_balance=best_params['normal_balance']
)

In [ ]:
import lightgbm as lgb
best_model = lgb.LGBMClassifier(
    boosting_type="gbdt",
    n_estimators=500,
    learning_rate=0.1,
    is_unbalance=True,
    subsample_freq=1,
    verbose=-1,
    metrics=["multi_error"],
    subsample=best_params["subsample"],
    colsample_bytree=best_params["feature_fraction"],
    reg_alpha=best_params["lambda_l1"],
    reg_lambda=best_params["lambda_l2"],
    num_leaves=best_params["num_leaves"],
)

In [ ]:
valores = []
def corrige_zero(label):
    # Converte para string caso não seja
    label_str = str(label)
    if (label_str.rstrip()=='0') or (label_str.rstrip()=='2'):
        return float(label_str.rstrip() + '.0')
    if (label_str.rstrip()=='102.'):
        return float(2.0)
    if (label_str.rstrip()=='101.'):
        return float(1.0)
    if (label_str.rstrip()=='105.'):
        return float(5.0)
    if (label_str.rstrip()=='106.'):
        return float(6.0)
    if (label_str.rstrip()=='108.'):
        return float(8.0)
    return float(label_str)
eventos = []
for evento in train_events:
    # Verifica se é uma Series
    if hasattr(evento['labels'], 'apply'):
        evento['labels'] = evento['labels'].apply(corrige_zero).fillna(0.0)
        valores.append(evento['labels'])
    eventos.append(evento)

In [ ]:
# valores = []
# def corrige_zero(label):
#     # Converte para string caso não seja
#     label_str = str(label)
#     if (label_str.rstrip()=='0'):
#         return float(label_str.rstrip() + '.0')
#     return float(label_str)
# eventos = []
# for evento in train_events:
#     # Verifica se é uma Series
#     if hasattr(evento['labels'], 'apply'):
#         evento['labels'] = evento['labels'].apply(corrige_zero).fillna(0.0)
#         valores.append(evento['labels'])
#     eventos.append(evento)

In [ ]:
transformed_train_events = MAEDataset.transform_events(
    train_events,
    best_experiment.raw_transform,
    instance_types=best_experiment.instance_types,
    tgt_events=best_experiment.tgt_events,
    n_jobs=-1,
)

full_train_set = MAEDataset.gather(transformed_train_events)

X = full_train_set.X
y = full_train_set.y
g = full_train_set.g
g_class = full_train_set.g_class
file = full_train_set.file

In [ ]:
file

In [ ]:
# X_list = []
# y_list = []
# event_types = []
# for evento in train_events:
#     X_evt, y_evt, evt_type = best_experiment.raw_transform(evento)
#     X_list.append(X_evt)
#     y_list.append(y_evt)
#     event_types.append(evt_type)
# # Agora, junte tudo em arrays únicos
# import numpy as np
# X = np.vstack(X_list)
# y = np.concatenate(y_list)

In [ ]:
best_experiment.fit(X, y)    # Fit preprocessors (scaler, imputer, pca, label_encoder)
X_transformed, y_transformed = best_experiment.transform(X, y)
best_model.fit(X_transformed, y_transformed)

In [ ]:
valores = []
def corrige_zero(label):
    # Converte para string caso não seja
    label_str = label
    if (label_str==102.0):
        return float(2.0)
    if (label_str==101.0):
        return float(1.0)
    if (label_str==105.0):
        return float(5.0)
    if (label_str==106.0):
        return float(6.0)
    if (label_str==108.0):
        return float(8.0)
    return float(label_str)
eventos = []
for evento in test_events:
    # Verifica se é uma Series
    if hasattr(evento['labels'], 'apply'):
        evento['labels'] = evento['labels'].apply(corrige_zero).fillna(0.0)
        valores.append(evento['labels'])
    eventos.append(evento)

In [ ]:
valores = []
def corrige_zero(label):
    # Converte para string caso não seja
    label_str = str(label)
    if (label_str.rstrip()=='0') or (label_str.rstrip()=='2'):
        return float(label_str.rstrip() + '.0')
    if (label_str.rstrip()=='102.'):
        return float(2.0)
    if (label_str.rstrip()=='101.'):
        return float(1.0)
    if (label_str.rstrip()=='105.'):
        return float(5.0)
    if (label_str.rstrip()=='106.'):
        return float(6.0)
    if (label_str.rstrip()=='108.'):
        return float(8.0)
    return float(label_str)
eventos = []
for evento in test_events:
    # Verifica se é uma Series
    if hasattr(evento['labels'], 'apply'):
        evento['labels'] = evento['labels'].apply(corrige_zero).fillna(0.0)
        valores.append(evento['labels'])
    eventos.append(evento)

In [ ]:
transformed_test_events = MAEDataset.transform_events(
    test_events,
    best_experiment.raw_transform,
    instance_types=best_experiment.instance_types,
    tgt_events=best_experiment.tgt_events,
    n_jobs=-1,
)

full_test_set = MAEDataset.gather(transformed_test_events)

X_test = full_test_set.X
y_test = full_test_set.y
g = full_test_set.g
g_class = full_test_set.g_class
file = full_test_set.file

In [ ]:
# 1. Máscara dos arquivos
file_mask = np.char.find(full_test_set.file.astype(str), 'WELL') >= 0
# 2. Índices dos arquivos
target_file_indices = np.where(file_mask)[0]
# 3. Máscara para g
mask = np.isin(full_test_set.g, target_file_indices)
# 4. Filtrando X e y
X_test = full_test_set.X[mask]
y_test = full_test_set.y[mask]

In [ ]:
type(train_events)

In [ ]:
np.unique(y)

In [ ]:
y_test = [0 if x == 9 else x for x in y_test]

In [ ]:
# X_list = []
# y_list = []
# event_types = []
# for evento in test_events:
#     X_evt, y_evt, evt_type = best_experiment.raw_transform(evento)
#     X_list.append(X_evt)
#     y_list.append(y_evt)
#     event_types.append(evt_type)Q@1sxzç~;......................111111111111111111111111111111111111111111111
# import numpy as np
# X_test = np.vstack(X_list)
# y_test = np.concatenate(y_list)

In [ ]:
X_transformed, y_transformed = best_experiment.transform(X_test, y_test)
y_predict = best_model.predict(X_transformed)

In [ ]:
np.unique(y)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
# Calcule a matriz de confusão bruta
cm = confusion_matrix(y_transformed, y_predict)
# Normalizar por linha (classe verdadeira)
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
plt.figure(figsize=(8, 6))
sns.heatmap(cm_percent, annot=True, fmt='.1f', cmap='Blues')
plt.xlabel('Classe Predita')
plt.ylabel('Classe Verdadeira')
plt.title('Matriz de Confusão (% por classe)')
plt.show()

In [ ]:
diagonal = np.diag(cm_percent)
media_diagonal = diagonal.mean()
print(f"Média dos valores na diagonal: {media_diagonal:.2f}%")

In [ ]:
full_test_set

In [ ]:
test_events

In [ ]:
train_events['tags']

In [ ]:
df = pd.DataFrame(train_events)

In [ ]:

df2 = pd.DataFrame(transformed_test_events)

In [ ]:

np.unique(g_class)

In [ ]:
df2[0][1]

In [ ]:
df2[1][1]

In [ ]:
a

In [ ]:
for idx, val in df2[1][1].items():
    print(idx, val)

In [ ]:
df2[1]

In [ ]:
np.unique(y)

In [ ]:
g = full_train_set.g
g_class = full_train_set.g_class

In [ ]:
full_train_set

In [ ]:
g[649]

In [ ]:
g[650]

In [ ]:
g_class[651]

In [ ]:
y[650]

In [ ]:
df2.index

In [ ]:
df2[2].unique()

In [ ]:
df[df['event_type']==8].unique_values()

In [ ]:
df['event_type'].unique()